# Solutions to Select Homework Exercises

In [1]:
# This is a code cell that imports the necessary libraries for our session.
import numpy as np                        # NumPy for numerical computations
import scipy as sp                        # SciPy for scientific computing
import sympy as sym                       # SymPy for symbolic mathematics
import matplotlib as mpl                  # Matplotlib for plotting
import matplotlib.pyplot as plt           # Matplotlib pyplot interface
from scipy.integrate import solve_ivp     # ODE solver
mpl.rcParams['figure.dpi'] = 150
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.right'] = False

This page collects full solutions to select homework exercises. Each
solution is worked out “by hand” and then, where helpful, confirmed
symbolically using SymPy or other computational tools.

## Homework 1

### Section 1.1 Exercise 9, page 10

Find two values of $\lambda$ for which $x(t) = e^{\lambda t}$ is a
solution of the differential equation $2x'' - 5x' - 3x = 0$.

#### By Hand

**Step 1 — Differentiate the candidate solution.**

For $x(t) = e^{\lambda t}$, $$
x'(t) = \lambda e^{\lambda t}, \qquad x''(t) = \lambda^2 e^{\lambda t}.
$$

**Step 2 — Substitute into the differential equation.**

$$
2x'' - 5x' - 3x = 2\lambda^2 e^{\lambda t} - 5\lambda e^{\lambda t} - 3 e^{\lambda t} = \left(2\lambda^2 - 5\lambda - 3\right)e^{\lambda t}.
$$

Since $e^{\lambda t} \neq 0$ for any $t$, this expression is zero
exactly when

$$
2\lambda^2 - 5\lambda - 3 = 0.
$$

This is the **characteristic equation** associated with the ODE.

**Step 3 — Solve the characteristic equation.**

Factoring, $$
2\lambda^2 - 5\lambda - 3 = (2\lambda + 1)(\lambda - 3) = 0,
$$ so $$
\lambda = -\frac{1}{2} \qquad \text{or} \qquad \lambda = 3.
$$

> **Tip**
>
> Both values work independently: $x_1(t) = e^{-t/2}$ and
> $x_2(t) = e^{3t}$ are each solutions of $2x'' - 5x' - 3x = 0$, and so
> is any linear combination $x(t) = c_1 e^{-t/2} + c_2 e^{3t}$.

#### Using SymPy

In [2]:
t, lam = sym.symbols('t lambda', real=True)

x = sym.exp(lam * t)

lhs = 2 * sym.diff(x, t, 2) - 5 * sym.diff(x, t) - 3 * x
char_eq = sym.simplify(lhs / x)   # divide out the common factor e^{lambda t}

print("2x'' - 5x' - 3x  simplifies to:", sym.expand(char_eq), " * e^(lambda t)")

lambda_solutions = sym.solve(sym.Eq(char_eq, 0), lam)
print("Values of lambda:", lambda_solutions)

2x'' - 5x' - 3x  simplifies to: 2*lambda**2 - 5*lambda - 3  * e^(lambda t)
Values of lambda: [-1/2, 3]

> **Note**
>
> SymPy confirms the two characteristic roots found by hand:
> $\lambda = -\dfrac{1}{2}$ and $\lambda = 3$.

------------------------------------------------------------------------

### Section 1.1 Exercise 12, page 11

*(Physics)* In deep water, the intensity of light $I = I(x)$ at a depth
$x$ meters below the water surface is modeled by the equation
$I' = -1.4I$. At what depth is the light intensity $1\%$ that at the
surface?

#### By Hand

**Step 1 — Solve the differential equation.**

The equation $I' = -1.4I$ is of the form $I'=aI$ where $a = -1.4$, so we
know from lecture that the general solution must be $I(x) = I_0 e^{ax}$
for some constant $I_0$.

**Step 2 — Impose the $1\%$ condition.**

We want the depth $x$ at which $I(x) = 0.01\,I_0$: $$
0.01\,I_0 = I_0 e^{-1.4x} \quad \Longrightarrow \quad 0.01 = e^{-1.4x}.
$$

**Step 3 — Solve for $x$.**

Taking the natural log of both sides, $$
\ln(0.01) = -1.4x \quad \Longrightarrow \quad x = -\frac{\ln(0.01)}{1.4} = \frac{\ln(100)}{1.4}.
$$

Numerically, $$
x = \frac{\ln(100)}{1.4} \approx \frac{4.6052}{1.4} \approx 3.29 \text{ meters}.
$$

> **Tip**
>
> **Interpretation.** Since $1.4 > 0$ is the decay rate, light intensity
> decreases exponentially with depth. At roughly $3.29$ meters below the
> surface, only $1\%$ of the surface intensity remains — a useful
> benchmark for how quickly light is absorbed in deep water.

## Homework 2

### Section 1.1.3 — Slope Fields

#### Exercise 6, Page 6

Use software to sketch the slope field for the differential equation
$x' = x^2 - t$ on the square $-3 < t < 3,\ -3 < x < 3$.

**Setting up the slope field.**

A slope field is produced by evaluating the right-hand side
$f(t,x) = x^2 - t$ at a grid of points $(t,x)$ in the given square, and
at each point drawing a short line segment with that slope. Since no
closed-form solution is needed, the natural way to “solve” this exercise
is with code.

In [3]:
def f(t, x):
    return x**2 - t

# Grid of points for the slope field arrows
t_grid = np.linspace(-3, 3, 21)
x_grid = np.linspace(-3, 3, 21)
T, X = np.meshgrid(t_grid, x_grid)
slopes = f(T, X)

# Normalize direction vectors so every arrow has the same visual length
dt = np.ones_like(slopes)
dx = slopes
norm = np.sqrt(dt**2 + dx**2)
dt_unit, dx_unit = dt / norm, dx / norm

fig, ax = plt.subplots(figsize=(6, 6))
ax.quiver(T, X, dt_unit, dx_unit, angles='xy', pivot='mid',
          headwidth=0, headlength=0, headaxislength=0,
          color='steelblue', width=0.003)

# Overlay a handful of numerically computed solution curves
for x0 in [-2.5, -1, 0, 1, 2.5]:
    sol = solve_ivp(f, [0, 3], [x0], dense_output=True, max_step=0.05)
    ax.plot(sol.t, sol.y[0], color='darkorange', lw=1.5)
    sol_back = solve_ivp(f, [0, -3], [x0], dense_output=True, max_step=0.05)
    ax.plot(sol_back.t, sol_back.y[0], color='darkorange', lw=1.5)

ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.set_xlabel('$t$', fontsize=13)
ax.set_ylabel('$x$', fontsize=13)
ax.set_title(r"Slope field for $x' = x^2 - t$", fontsize=13)
plt.tight_layout()
plt.show()

> **Note**
>
> There is no elementary closed-form solution for $x' = x^2 - t$ (it is
> a form of **Riccati equation**), so this exercise is intentionally
> solved by producing the slope field and, optionally, a few
> numerically-generated solution curves rather than an explicit formula.

------------------------------------------------------------------------

### Section 1.3.1 — Separable Equations

#### Exercise 4(a), Page 26

Find the general solution: $x' = \dfrac{2x}{t+1}$.

**By Hand.**

Separate variables (assuming $x \neq 0$ and $t \neq -1$): $$
\frac{dx}{x} = \frac{2}{t+1}\,dt.
$$

Integrating both sides, $$
\ln|x| = 2\ln|t+1| + C.
$$

Exponentiating, $$
|x| = e^{C}(t+1)^2 \quad \Longrightarrow \quad x(t) = A(t+1)^2,
$$ where $A$ is an arbitrary constant (absorbing the sign and $e^C$).
Note $A = 0$ recovers the trivial solution $x \equiv 0$, so this formula
captures **all** solutions.

$$
\boxed{x(t) = A(t+1)^2}
$$

In [4]:
t = sym.symbols('t')
x = sym.Function('x')

eq = sym.Eq(sym.diff(x(t), t), 2*x(t)/(t+1))
sol = sym.dsolve(eq, x(t))
print("General solution:", sol)

General solution: Eq(x(t), C1*(t**2 + 2*t + 1))

> **Note**
>
> SymPy returns $x(t) = C_1(t+1)^2$, matching the boxed formula above
> (with $A = C_1$).

#### Exercise 10, Page 27

Solve the following initial value problems.

##### (a) $\dfrac{dx}{dt} = e^{t+x}, \quad x(0) = 0$.

**By Hand.**

Rewrite the right-hand side as $e^t e^x$ and separate variables: $$
e^{-x}\,dx = e^{t}\,dt.
$$

Integrating, $$
-e^{-x} = e^{t} + C.
$$

Apply $x(0) = 0$:
$-e^{0} = e^{0} + C \Rightarrow -1 = 1 + C \Rightarrow C = -2$. So $$
-e^{-x} = e^t - 2 \quad \Longrightarrow \quad e^{-x} = 2 - e^t \quad \Longrightarrow \quad x(t) = -\ln\!\left(2 - e^t\right).
$$

$$
\boxed{x(t) = -\ln(2 - e^t)}, \qquad t < \ln 2.
$$

> **Tip**
>
> The restriction $t < \ln 2$ is required for $2 - e^t > 0$, so that the
> logarithm is defined. This is an example of a solution whose
> **interval of existence** is limited even though the differential
> equation itself is defined for all $t$ and $x$.

In [5]:
xf = sym.Function('x')
eq10a = sym.Eq(sym.diff(xf(t), t), sym.exp(t + xf(t)))
sol10a = sym.dsolve(eq10a, xf(t), ics={xf(0): 0})
print("Solution:", sym.simplify(sol10a))

Solution: Eq(x(t), log(-1/(exp(t) - 2)))

##### (b) $\dfrac{dT}{dt} = 2at\left(T^2 - a^2\right), \quad T(0) = 0$.

**By Hand.**

Separate variables: $$
\frac{dT}{T^2 - a^2} = 2at\,dt.
$$

Using the partial fraction decomposition
$\dfrac{1}{T^2-a^2} = \dfrac{1}{2a}\left(\dfrac{1}{T-a} - \dfrac{1}{T+a}\right)$
and integrating both sides, $$
\frac{1}{2a}\ln\left|\frac{T-a}{T+a}\right| = at^2 + C.
$$

Applying $T(0) = 0$: the left side is
$\dfrac{1}{2a}\ln\left|\dfrac{-a}{a}\right| = \dfrac{1}{2a}\ln(1) = 0$,
so $C = 0$. Thus $$
\ln\left|\frac{T-a}{T+a}\right| = 2a^2t^2 \quad \Longrightarrow \quad \frac{T-a}{T+a} = \pm\, e^{2a^2t^2}.
$$

The initial condition $T(0)=0$ forces
$\dfrac{T-a}{T+a}\Big|_{t=0} = -1$, so the sign is negative: $$
\frac{T-a}{T+a} = -e^{2a^2t^2}.
$$

Solving for $T$: $$
T - a = -e^{2a^2t^2}(T+a) \quad \Longrightarrow \quad T\left(1+e^{2a^2t^2}\right) = a\left(1 - e^{2a^2t^2}\right)
$$ $$
\Longrightarrow \quad T = a\,\frac{1 - e^{2a^2t^2}}{1+e^{2a^2t^2}} = -a\,\frac{e^{2a^2t^2}-1}{e^{2a^2t^2}+1}.
$$

Using the identity $\tanh(u) = \dfrac{e^{2u}-1}{e^{2u}+1}$ with
$u = a^2t^2$, this simplifies to

$$
\boxed{T(t) = -a\tanh\!\left(a^2t^2\right)}.
$$

In [6]:
a, t = sym.symbols('a t', positive=True)
Tf = sym.Function('T')

# Verify the closed-form solution directly, since dsolve with these ics
# runs into the usual difficulty of picking out one branch of a
# multivalued implicit solution.
T_guess = -a * sym.tanh(a**2 * t**2)

lhs = sym.diff(T_guess, t)
rhs = 2*a*t*(T_guess**2 - a**2)
print("Residual dT/dt - 2at(T^2 - a^2):", sym.simplify(lhs - rhs))
print("T(0) =", T_guess.subs(t, 0))

Residual dT/dt - 2at(T^2 - a^2): 0
T(0) = 0

> **Note**
>
> SymPy confirms both that the proposed closed-form
> $T(t) = -a\tanh(a^2t^2)$ satisfies the differential equation exactly
> (the residual simplifies to $0$) and that it meets the initial
> condition $T(0) = 0$.

##### (c) $\dfrac{dy}{dt} = t^2\tan y, \quad y(0) = 0$.

**By Hand.**

Notice that $y \equiv 0$ makes both sides zero: the left side is
$\frac{d}{dt}(0) = 0$, and the right side is $t^2\tan(0) = 0$. Since
$y(0) = 0$ matches this constant function, and the right-hand side
$t^2\tan y$ is smooth (hence Lipschitz) near $y = 0$, the initial value
problem has a **unique** solution — and it must be the equilibrium
solution.

$$
\boxed{y(t) \equiv 0}
$$

> **Tip**
>
> It is instructive to see where a naive separation of variables leads:
> dividing by $\tan y$ gives $\cot y\,dy = t^2\,dt$, so
> $\ln|\sin y| = \tfrac{1}{3}t^3 + C$. This step implicitly assumes
> $\sin y \neq 0$, silently discarding the constant solution
> $y \equiv 0$ — exactly the solution the initial condition selects.
> This is a good reminder to always check for equilibrium (constant)
> solutions before dividing them away.

In [7]:
t = sym.symbols('t')
yf = sym.Function('y')
eq10c = sym.Eq(sym.diff(yf(t), t), t**2 * sym.tan(yf(t)))

# The zero function should satisfy the ODE identically
residual = sym.diff(0, t) - t**2 * sym.tan(0)
print("Residual for y = 0:", residual)

Residual for y = 0: 0

------------------------------------------------------------------------

### Section 1.4.1 — First-Order Linear Equations & Integrating Factors

#### Exercise 2(a), Page 42

Find the general solution: $x' = -\dfrac{2}{t}x + t$.

**By Hand.**

Write the equation in standard linear form: $$
x' + \frac{2}{t}x = t.
$$

The integrating factor is $$
\mu(t) = \exp\!\left(\int \frac{2}{t}\,dt\right) = \exp\!\left(2\ln|t|\right) = t^2.
$$

Multiplying both sides by $\mu(t) = t^2$, the left side becomes an exact
derivative: $$
\left(t^2 x\right)' = t^2 \cdot t = t^3.
$$

Integrating, $$
t^2 x = \frac{t^4}{4} + C.
$$

$$
\boxed{x(t) = \frac{t^2}{4} + \frac{C}{t^2}}
$$

In [8]:
t = sym.symbols('t')
x = sym.Function('x')

eq2a = sym.Eq(sym.diff(x(t), t), -(2/t)*x(t) + t)
sol2a = sym.dsolve(eq2a, x(t))
print("General solution:", sol2a)

General solution: Eq(x(t), (C1 + t**4/4)/t**2)

> **Note**
>
> SymPy’s result
> $x(t) = \dfrac{C_1 + t^4/4}{t^2} = \dfrac{C_1}{t^2} + \dfrac{t^2}{4}$
> agrees with the boxed formula.

#### Exercise 3(a), Page 42

Solve the initial value problem:
$x' + \dfrac{5}{t}x = 1+t, \quad x(1) = 1$.

**By Hand.**

The equation is already in standard linear form with $p(t) = 5/t$. The
integrating factor is $$
\mu(t) = \exp\!\left(\int \frac{5}{t}\,dt\right) = e^{5\ln|t|} = t^5.
$$

Multiplying through by $t^5$: $$
\left(t^5 x\right)' = t^5(1+t) = t^5 + t^6.
$$

Integrating, $$
t^5 x = \frac{t^6}{6} + \frac{t^7}{7} + C \quad \Longrightarrow \quad x(t) = \frac{t}{6} + \frac{t^2}{7} + \frac{C}{t^5}.
$$

Applying $x(1) = 1$: $$
1 = \frac{1}{6} + \frac{1}{7} + C \quad \Longrightarrow \quad C = 1 - \frac{13}{42} = \frac{29}{42}.
$$

$$
\boxed{x(t) = \frac{t}{6} + \frac{t^2}{7} + \frac{29}{42\,t^5}}
$$

In [9]:
t = sym.symbols('t')
x = sym.Function('x')

eq3a = sym.Eq(sym.diff(x(t), t) + (5/t)*x(t), 1 + t)
sol3a = sym.dsolve(eq3a, x(t), ics={x(1): 1})
print("Particular solution:", sym.simplify(sol3a))

# Compare against the boxed formula
manual = t/6 + t**2/7 + sym.Rational(29, 42)/t**5
residual = sym.simplify(manual - sol3a.rhs)
print("Difference from by-hand formula:", residual)

Particular solution: Eq(x(t), (t**6*(6*t + 7) + 29)/(42*t**5))
Difference from by-hand formula: 0

> **Note**
>
> SymPy’s particular solution agrees exactly with the by-hand formula
> (the difference simplifies to $0$).

------------------------------------------------------------------------

### Section 1.4.3 — Applications: RC Circuits

#### Exercise 1, Page 53

Write the equation that governs an RC circuit with a 12-volt battery,
taking $R = 1$ and $C = \tfrac{1}{2}$. Determine the equilibrium
solution and its stability. If $Q(0) = 5$, find a formula for $Q(t)$.
Find the current $I(t)$. Plot the charge and the current on the same set
of axes.

**By Hand.**

**Step 1 — Set up the governing equation.**

For an RC circuit with a constant voltage source $V$, resistance $R$,
and capacitance $C$, the charge $Q(t)$ on the capacitor satisfies $$
R\frac{dQ}{dt} + \frac{Q}{C} = V \quad \Longrightarrow \quad \frac{dQ}{dt} = \frac{V}{R} - \frac{Q}{RC}.
$$

Substituting $V = 12$, $R = 1$, $C = \tfrac12$: $$
\frac{dQ}{dt} = 12 - 2Q.
$$

**Step 2 — Find the equilibrium solution and its stability.**

Setting $\dfrac{dQ}{dt} = 0$ gives $12 - 2Q_{\text{eq}} = 0$, so
$Q_{\text{eq}} = 6$. Writing the equation as $$
\frac{dQ}{dt} = -2(Q - 6),
$$ we see that $Q > 6 \Rightarrow \dfrac{dQ}{dt} < 0$ and
$Q < 6 \Rightarrow \dfrac{dQ}{dt} > 0$: solutions are always pushed back
toward $Q = 6$. Hence $Q_{\text{eq}} = 6$ is a **stable** equilibrium.

**Step 3 — Solve for $Q(t)$ with $Q(0) = 5$.**

This is linear with integrating factor $\mu(t) = e^{2t}$: $$
\left(Qe^{2t}\right)' = 12e^{2t} \quad \Longrightarrow \quad Qe^{2t} = 6e^{2t} + C \quad \Longrightarrow \quad Q(t) = 6 + Ce^{-2t}.
$$

Applying $Q(0) = 5$: $5 = 6 + C \Rightarrow C = -1$.

$$
\boxed{Q(t) = 6 - e^{-2t}}
$$

**Step 4 — Find the current $I(t)$.**

The current is the rate of change of charge: $$
I(t) = \frac{dQ}{dt} = \frac{d}{dt}\left(6 - e^{-2t}\right).
$$

$$
\boxed{I(t) = 2e^{-2t}}
$$

In [10]:
t = sym.symbols('t')
Q = sym.Function('Q')

eqQ = sym.Eq(sym.diff(Q(t), t), 12 - 2*Q(t))
solQ = sym.dsolve(eqQ, Q(t), ics={Q(0): 5})
print("Q(t):", solQ)

I_expr = sym.diff(solQ.rhs, t)
print("I(t) = dQ/dt:", I_expr)

Q(t): Eq(Q(t), 6 - exp(-2*t))
I(t) = dQ/dt: 2*exp(-2*t)

> **Note**
>
> SymPy confirms $Q(t) = 6 - e^{-2t}$, and differentiating gives
> $I(t) = 2e^{-2t}$, matching both boxed results.

**Step 5 — Plot $Q(t)$ and $I(t)$.**

In [11]:
t_vals = np.linspace(0, 4, 400)
Q_vals = 6 - np.exp(-2*t_vals)
I_vals = 2*np.exp(-2*t_vals)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(t_vals, Q_vals, color='steelblue', lw=2, label='$Q(t)$ (charge)')
ax.plot(t_vals, I_vals, color='darkorange', lw=2, label='$I(t)$ (current)')
ax.axhline(6, color='gray', linestyle='--', lw=1, label='Equilibrium $Q_{eq} = 6$')

ax.set_xlabel('$t$', fontsize=13)
ax.set_ylabel('Charge / Current', fontsize=13)
ax.set_title('RC Circuit: Charge and Current vs. Time', fontsize=13)
ax.legend(fontsize=10, loc='center right')
plt.tight_layout()
plt.show()

## Homework 3

### Section 1.5.1 — Autonomous Systems

#### Exercise 2(b)

For the model $x' = x(4-x)(5-x)^2$: (a) plot the growth rate $f(x)$
versus $x$ and sketch the phase line diagram; (b) find the equilibria
analytically and classify them according to their stability using
Theorem 1.32; (c) draw a few key time series plots, $x = x(t)$, in the
$tx$-plane.

**(a) & (b) — Equilibria, stability, and the phase line.**

Setting $f(x) = x(4-x)(5-x)^2 = 0$ gives three equilibria:
$x = 0,\ 4,\ 5$.

Theorem 1.32 classifies a *hyperbolic* equilibrium (one where
$f'(x_{eq}) \neq 0$) by the sign of $f'(x_{eq})$: negative means stable,
positive means unstable. Differentiating and evaluating, $$
f'(0) = 100 > 0 \ (\text{unstable}), \qquad f'(4) = -4 < 0 \ (\text{stable}), \qquad f'(5) = 0 \ (\text{inconclusive}).
$$

Since $f'(5) = 0$, Theorem 1.32 does not directly apply at $x=5$ and we
examine the sign of $f(x)$ directly instead. Because $(5-x)^2 \geq 0$
never changes sign, the sign of $f(x)$ near $x=5$ is controlled by
$x(4-x)$, which is negative for $x$ near $5$ on **both** sides.
Tabulating the sign of $f$ on each interval:

| Interval       | $(-\infty, 0)$ | $(0,4)$ | $(4,5)$ | $(5,\infty)$ |
|----------------|----------------|---------|---------|--------------|
| sign of $f(x)$ | $-$            | $+$     | $-$     | $-$          |

- At $x=0$: sign goes $- \to +$, so trajectories are pushed away on both
  sides $\Rightarrow$ **unstable**.
- At $x=4$: sign goes $+ \to -$, so trajectories are pulled in from both
  sides $\Rightarrow$ **stable**.
- At $x=5$: the sign is negative on both sides, so trajectories from
  below move *away* from $5$ (down toward $4$) while trajectories from
  above move *toward* $5$ $\Rightarrow$ **semistable**.

In [12]:
def f2b(x):
    return x*(4-x)*(5-x)**2

xmin, xmax = -1, 6
x = np.linspace(xmin, xmax, 500)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 6),
                                gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

ax1.plot(x, f2b(x), color='steelblue', lw=2)
ax1.axhline(0, color='black', lw=1)
ax1.plot(0, 0, marker='o', ms=10, mfc='white', mec='crimson', mew=2, zorder=5)          # unstable
ax1.plot(4, 0, marker='o', ms=10, mfc='seagreen', mec='seagreen', zorder=5)             # stable
ax1.plot(5, 0, marker='o', ms=10, fillstyle='right', mfc='seagreen',
          markerfacecoloralt='crimson', mec='black', zorder=5)                          # semistable
ax1.set_ylim(-60, 60)
ax1.set_ylabel('$f(x)$')
ax1.set_title(r"Growth rate $f(x) = x(4-x)(5-x)^2$")

bounds = [xmin, 0, 4, 5, xmax]
ax2.axhline(0, color='black', lw=1.5)
for a, b in zip(bounds[:-1], bounds[1:]):
    mid = (a + b) / 2
    sign = np.sign(f2b(mid))
    x0, x1 = (a + 0.15, b - 0.15) if sign > 0 else (b - 0.15, a + 0.15)
    ax2.annotate('', xy=(x1, 0), xytext=(x0, 0),
                 arrowprops=dict(arrowstyle='-|>', color='darkorange', lw=2))
ax2.plot(0, 0, marker='o', ms=10, mfc='white', mec='crimson', mew=2, zorder=5)
ax2.plot(4, 0, marker='o', ms=10, mfc='seagreen', mec='seagreen', zorder=5)
ax2.plot(5, 0, marker='o', ms=10, fillstyle='right', mfc='seagreen',
          markerfacecoloralt='crimson', mec='black', zorder=5)
ax2.set_yticks([])
ax2.set_xlabel('$x$')
ax2.set_ylim(-1, 1)

plt.tight_layout()
plt.show()

In [13]:
x = sym.symbols('x')
f = x*(4-x)*(5-x)**2

eqs = sym.solve(sym.Eq(f, 0), x)
fp = sym.diff(f, x)
print("Equilibria:", eqs)
for e in eqs:
    print(f"  f'({e}) =", fp.subs(x, e))

Equilibria: [0, 4, 5]
  f'(0) = 100
  f'(4) = -4
  f'(5) = 0

> **Note**
>
> SymPy confirms the equilibria $x = 0, 4, 5$ and the derivative values
> used above; the vanishing derivative at $x=5$ is exactly what forced
> the direct sign analysis.

**(c) — Representative time series.**

In [14]:
def f2b_t(t, x):
    return x*(4-x)*(5-x)**2

fig, ax = plt.subplots(figsize=(7, 5))
for x0 in [-0.5, 0.5, 1, 3, 4.5, 5, 5.5, 6]:
    sol = solve_ivp(f2b_t, [0, 3], [x0], dense_output=True, max_step=0.01, rtol=1e-8)
    ax.plot(sol.t, sol.y[0], lw=2, label=f'$x_0={x0}$')

for level in [0, 4, 5]:
    ax.axhline(level, color='gray', linestyle='--', lw=1)

ax.set_ylim(-3, 7)
ax.set_xlabel('$t$')
ax.set_ylabel('$x(t)$')
ax.set_title(r"Representative solutions for $x' = x(4-x)(5-x)^2$")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

------------------------------------------------------------------------

#### Exercise 2(g)

For the model $x' = (4-x)(2-x)^3$: (a) plot the growth rate $f(x)$
versus $x$ and sketch the phase line diagram; (b) find the equilibria
analytically and classify them according to their stability using
Theorem 1.32; (c) draw a few key time series plots, $x = x(t)$, in the
$tx$-plane.

**(a) & (b) — Equilibria, stability, and the phase line.**

Setting $f(x) = (4-x)(2-x)^3 = 0$ gives two equilibria: $x = 2$ (a
triple root of the factor $(2-x)^3$) and $x = 4$.

$$
f'(2) = 0 \ (\text{inconclusive}), \qquad f'(4) = 8 > 0 \ (\text{unstable}).
$$

Since $f'(2)=0$ (the triple root makes $x=2$ non-hyperbolic), we again
check signs directly:

| Interval       | $(-\infty, 2)$ | $(2,4)$ | $(4,\infty)$ |
|----------------|----------------|---------|--------------|
| sign of $f(x)$ | $+$            | $-$     | $+$          |

- At $x=2$: sign goes $+ \to -$ $\Rightarrow$ **stable**.
- At $x=4$: sign goes $- \to +$, consistent with $f'(4)>0$ $\Rightarrow$
  **unstable**.

In [15]:
def f2g(x):
    return (4-x)*(2-x)**3

xmin, xmax = 0, 6
x = np.linspace(xmin, xmax, 500)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 6),
                                gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

ax1.plot(x, f2g(x), color='steelblue', lw=2)
ax1.axhline(0, color='black', lw=1)
ax1.plot(2, 0, marker='o', ms=10, mfc='seagreen', mec='seagreen', zorder=5)     # stable
ax1.plot(4, 0, marker='o', ms=10, mfc='white', mec='crimson', mew=2, zorder=5) # unstable
ax1.set_ylim(-30, 30)
ax1.set_ylabel('$f(x)$')
ax1.set_title(r"Growth rate $f(x) = (4-x)(2-x)^3$")

bounds = [xmin, 2, 4, xmax]
ax2.axhline(0, color='black', lw=1.5)
for a, b in zip(bounds[:-1], bounds[1:]):
    mid = (a + b) / 2
    sign = np.sign(f2g(mid))
    x0, x1 = (a + 0.15, b - 0.15) if sign > 0 else (b - 0.15, a + 0.15)
    ax2.annotate('', xy=(x1, 0), xytext=(x0, 0),
                 arrowprops=dict(arrowstyle='-|>', color='darkorange', lw=2))
ax2.plot(2, 0, marker='o', ms=10, mfc='seagreen', mec='seagreen', zorder=5)
ax2.plot(4, 0, marker='o', ms=10, mfc='white', mec='crimson', mew=2, zorder=5)
ax2.set_yticks([])
ax2.set_xlabel('$x$')
ax2.set_ylim(-1, 1)

plt.tight_layout()
plt.show()

In [16]:
x = sym.symbols('x')
f = (4-x)*(2-x)**3

eqs = sym.solve(sym.Eq(f, 0), x)
fp = sym.diff(f, x)
print("Equilibria:", eqs)
for e in eqs:
    print(f"  f'({e}) =", fp.subs(x, e))

Equilibria: [2, 4]
  f'(2) = 0
  f'(4) = 8

**(c) — Representative time series.**

In [17]:
def f2g_t(t, x):
    return (4-x)*(2-x)**3

fig, ax = plt.subplots(figsize=(7, 5))
for x0 in [0.5, 1.5, 2, 3, 3.9, 4, 4.1, 4.5]:
    sol = solve_ivp(f2g_t, [0, 2], [x0], dense_output=True, max_step=0.005, rtol=1e-9)
    ax.plot(sol.t, sol.y[0], lw=2, label=f'$x_0={x0}$')

for level in [2, 4]:
    ax.axhline(level, color='gray', linestyle='--', lw=1)

ax.set_ylim(0, 8)
ax.set_xlabel('$t$')
ax.set_ylabel('$x(t)$')
ax.set_title(r"Representative solutions for $x' = (4-x)(2-x)^3$")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

> **Tip**
>
> Notice how flat the curves are near $x=2$ compared with how sharply
> solutions bend away near $x=4$. This reflects $f'(2)=0$ versus
> $f'(4) = 8$: convergence toward a non-hyperbolic equilibrium (a
> repeated root) is much slower than the exponential rate predicted by
> Theorem 1.32 at a hyperbolic one.

------------------------------------------------------------------------

#### Exercise 3(c)

Use analytical or graphical methods to determine the equilibria of
$R' = \dfrac{3R}{1+R^2} - 1$.

**By Hand.**

Setting $f(R) = \dfrac{3R}{1+R^2} - 1 = 0$: $$
\frac{3R}{1+R^2} = 1 \quad \Longrightarrow \quad 3R = 1 + R^2 \quad \Longrightarrow \quad R^2 - 3R + 1 = 0.
$$

By the quadratic formula, $$
R = \frac{3 \pm \sqrt{9-4}}{2} = \frac{3 \pm \sqrt{5}}{2}.
$$

$$
\boxed{R_1 = \frac{3-\sqrt{5}}{2} \approx 0.382, \qquad R_2 = \frac{3+\sqrt{5}}{2} \approx 2.618}
$$

Graphically, these are exactly the two places where the curve $f(R)$
crosses the $R$-axis:

In [18]:
def f3c(R):
    return 3*R/(1+R**2) - 1

R = np.linspace(-1, 6, 500)
R1 = (3 - np.sqrt(5)) / 2
R2 = (3 + np.sqrt(5)) / 2

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(R, f3c(R), color='steelblue', lw=2)
ax.axhline(0, color='black', lw=1)
ax.plot(R1, 0, marker='o', ms=10, mfc='white', mec='crimson', mew=2, zorder=5)
ax.plot(R2, 0, marker='o', ms=10, mfc='seagreen', mec='seagreen', zorder=5)
ax.annotate(f'$R_1 \\approx {R1:.3f}$', xy=(R1, 0), xytext=(R1-0.3, 0.6),
            fontsize=9, ha='right', arrowprops=dict(arrowstyle='->', lw=1))
ax.annotate(f'$R_2 \\approx {R2:.3f}$', xy=(R2, 0), xytext=(R2+0.3, 0.6),
            fontsize=9, ha='left', arrowprops=dict(arrowstyle='->', lw=1))
ax.set_xlabel('$R$')
ax.set_ylabel("$R' = f(R)$")
ax.set_title(r"Growth rate $f(R) = \dfrac{3R}{1+R^2} - 1$")
plt.tight_layout()
plt.show()

In [19]:
R = sym.symbols('R')
f = 3*R/(1+R**2) - 1
eqs = sym.solve(sym.Eq(f, 0), R)
print("Equilibria (exact):", eqs)
print("Equilibria (decimal):", [sym.N(e) for e in eqs])

Equilibria (exact): [3/2 - sqrt(5)/2, sqrt(5)/2 + 3/2]
Equilibria (decimal): [0.381966011250105, 2.61803398874989]

> **Tip**
>
> Although the exercise only asks for the equilibria, stability comes
> almost for free from the same plot: the curve crosses from $+$ to $-$
> at $R_2$ (stable) and from $-$ to $+$ at $R_1$ (unstable) — consistent
> with $f'(R_1) \approx 1.95 > 0$ and $f'(R_2) \approx -0.28 < 0$.

------------------------------------------------------------------------

#### Exercise 6 (Heat transfer)

Heat transfer by radiation from a body to its surroundings is modeled by
the Stefan–Boltzmann law $$
\frac{dT}{dt} = -k\left(T^4 - S^4\right),
$$ where $T = T(t)$ is the absolute temperature of the body and $S$ is
the (constant) temperature of the surroundings.

**(a) Phase line diagram and nature of the solution.**

The only equilibrium (for $T, S \geq 0$) is $T = S$, since $T^4 = S^4$
with $T,S\ge 0$ forces $T=S$. Writing $f(T) = -k(T^4-S^4)$:

- For $T > S$: $T^4 > S^4$, so $f(T) < 0$ (temperature decreasing — the
  body cools toward $S$).
- For $T < S$: $T^4 < S^4$, so $f(T) > 0$ (temperature increasing — the
  body warms toward $S$).

So $T = S$ is a **stable** equilibrium, and every solution approaches it
*monotonically* (no overshoot or oscillation) — exactly the physically
expected behavior of a body radiating heat until it reaches thermal
equilibrium with its surroundings.

In [20]:
k_demo = 2.0e-12
S_demo = 300

def fT(T, S):
    return -k_demo*(T**4 - S**4)

Tvals = np.linspace(0, 2*S_demo, 300)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 5.5),
                                gridspec_kw={'height_ratios': [3, 1]}, sharex=True)
ax1.plot(Tvals, fT(Tvals, S_demo), color='steelblue', lw=2)
ax1.axhline(0, color='black', lw=1)
ax1.plot(S_demo, 0, marker='o', ms=10, mfc='seagreen', mec='seagreen', zorder=5)
ax1.set_ylabel("$T' = -k(T^4-S^4)$")
ax1.set_title("Growth rate for $T' = -k(T^4 - S^4)$")

bounds = [0, S_demo, 2*S_demo]
ax2.axhline(0, color='black', lw=1.5)
for a, b in zip(bounds[:-1], bounds[1:]):
    mid = (a + b) / 2
    sign = np.sign(fT(mid, S_demo))
    x0, x1 = (a + 0.05*S_demo, b - 0.05*S_demo) if sign > 0 else (b - 0.05*S_demo, a + 0.05*S_demo)
    ax2.annotate('', xy=(x1, 0), xytext=(x0, 0),
                 arrowprops=dict(arrowstyle='-|>', color='darkorange', lw=2))
ax2.plot(S_demo, 0, marker='o', ms=10, mfc='seagreen', mec='seagreen', zorder=5)
ax2.set_yticks([])
ax2.set_xlabel('$T$')
ax2.set_ylim(-1, 1)

plt.tight_layout()
plt.show()

**(b) Approximate solution neglecting $S$.**

With $T(0) = 2000\,\text{K}$ and $S = 300\,\text{K}$, since $S \ll T$ we
drop the $S^4$ term and solve $$
\frac{dT}{dt} = -kT^4, \qquad k = 2.0\times10^{-12}\ \text{K}^{-3}\text{sec}^{-1}.
$$

This is separable: $$
\frac{dT}{T^4} = -k\,dt \quad \Longrightarrow \quad -\frac{1}{3}T^{-3} = -kt + C \quad \Longrightarrow \quad T^{-3} = 3kt + T_0^{-3}.
$$

Solving for $T$ and substituting $T_0 = 2000$: $$
T(t) = \frac{T_0}{\left(1 + 3kT_0^3\,t\right)^{1/3}}.
$$

Since $3kT_0^3 = 3(2.0\times10^{-12})(2000)^3 = 0.048\ \text{sec}^{-1}$,

$$
\boxed{T(t) = \frac{2000}{(1+0.048\,t)^{1/3}} \ \text{K}, \qquad t \text{ in seconds}.}
$$

In [21]:
t = sym.symbols('t', positive=True)
T = sym.Function('T')
k_val = sym.Rational(2, 10**12)

eq = sym.Eq(sym.diff(T(t), t), -k_val*T(t)**4)
sol = sym.dsolve(eq, T(t), ics={T(0): 2000})
print("Solution:", sym.simplify(sol))
print("Coefficient 3*k*T0^3 =", sym.N(3*k_val*2000**3))

Solution: Eq(T(t), 10000/(6*t + 125)**(1/3))
Coefficient 3*k*T0^3 = 0.0480000000000000

> **Note**
>
> SymPy confirms
> $T(t)^{-3} = \dfrac{3t}{5\times10^{11}} + \dfrac{1}{8\times 10^9}$ (an
> equivalent way of writing the boxed formula), and the numerically
> computed coefficient matches $0.048$.

In [22]:
T0 = 2000
k = 2.0e-12
S = 300

t_vals = np.linspace(0, 3000, 400)
T_approx = T0 * (1 + 3*k*T0**3*t_vals)**(-1/3)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(t_vals, T_approx, color='steelblue', lw=2, label=r'$T(t) = 2000(1+0.048t)^{-1/3}$')
ax.axhline(S, color='gray', linestyle='--', lw=1, label='$S = 300$ K')
ax.set_xlabel('$t$ (seconds)')
ax.set_ylabel('$T$ (K)')
ax.set_title('Approximate cooling curve, neglecting $S$')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

> **Tip**
>
> The approximate solution is only valid while $T \gg S$. By
> $t \approx 3000$ sec the body has cooled to around $380\,$K — close
> enough to $S = 300\,$K that neglecting $S^4$ in the original equation
> is no longer well justified; the full equation (including $S^4$) would
> need to be solved (numerically) to accurately track the final approach
> to equilibrium.

------------------------------------------------------------------------

### Section 1.5.2 — Bifurcations

For each model below (containing a parameter $h$): (a) find the
equilibria in terms of $h$ and determine their stability using Theorem
1.32, to the extent possible; (b) construct a bifurcation diagram
showing how the equilibria depend on $h$, labeling the branches as
stable or unstable.

#### Exercise 1(a): $x' = h - x^3$

**By Hand.**

Setting $f(x) = h - x^3 = 0$ gives the unique real equilibrium $$
x_{eq}(h) = h^{1/3}
$$ (the cube root function is one-to-one on $\mathbb{R}$, so there is
exactly one real equilibrium for every $h$; the other two roots of
$x^3=h$ are complex and not relevant here).

Differentiating, $f'(x) = -3x^2$, so $$
f'\!\left(x_{eq}\right) = -3h^{2/3} \leq 0 \quad \text{for every } h,
$$ with equality only at $h=0$. Thus $x_{eq}(h) = h^{1/3}$ is **stable
for every $h \neq 0$**. At $h=0$ (so $x_{eq}=0$), Theorem 1.32 is
inconclusive since $f'(0)=0$, but checking $f(x,0) = -x^3$ directly
shows $f>0$ for $x<0$ and $f<0$ for $x>0$ — trajectories are still drawn
toward $0$ from both sides, so $x=0$ remains stable (just
non-hyperbolically so).

> **Note**
>
> There is no actual bifurcation here: a single, always-stable branch of
> equilibria exists for every value of $h$, and no qualitative change in
> the equilibrium structure ever occurs. This example is a useful
> contrast to (b)–(d) below, where genuine bifurcations do occur.

In [23]:
h = np.linspace(-4, 4, 400)
x_eq = np.sign(h) * np.abs(h)**(1/3)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(h, x_eq, color='seagreen', lw=2.5, label='stable')
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_xlabel('$h$')
ax.set_ylabel('$x_{eq}$')
ax.set_title(r"Bifurcation diagram: $x' = h - x^3$")
ax.legend()
plt.tight_layout()
plt.show()

------------------------------------------------------------------------

#### Exercise 1(b): $x' = (x-1)(x-h)$

**By Hand.**

The equilibria are $x=1$ and $x=h$. Differentiating,
$f'(x) = (x-h)+(x-1) = 2x-(1+h)$, so $$
f'(1) = 1-h, \qquad f'(h) = h-1.
$$

- $x=1$ is **stable** when $h>1$ (so $f'(1)<0$) and **unstable** when
  $h<1$.
- $x=h$ is **stable** when $h<1$ (so $f'(h)<0$) and **unstable** when
  $h>1$.

At $h=1$, the two equilibria collide at $x=1$ and *exchange stability*
as $h$ increases through $1$ — this is the hallmark of a **transcritical
bifurcation**.

In [24]:
h_sym, x_sym = sym.symbols('h x')
f = (x_sym-1)*(x_sym-h_sym)
eqs = sym.solve(sym.Eq(f,0), x_sym)
fp = sym.diff(f, x_sym)
print("Equilibria:", eqs)
for e in eqs:
    print(f"  f'({e}) =", sym.simplify(fp.subs(x_sym, e)))

Equilibria: [1, h]
  f'(1) = 1 - h
  f'(h) = h - 1

In [25]:
h = np.linspace(-3, 3, 400)
h_lo, h_hi = h[h <= 1], h[h >= 1]

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(h_lo, np.ones_like(h_lo), color='crimson', lw=2.5, linestyle='--')
ax.plot(h_hi, np.ones_like(h_hi), color='seagreen', lw=2.5)
ax.plot(h_lo, h_lo, color='seagreen', lw=2.5)
ax.plot(h_hi, h_hi, color='crimson', lw=2.5, linestyle='--')
ax.plot(1, 1, marker='o', mfc='black', mec='black', ms=6)

ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_xlabel('$h$'); ax.set_ylabel('$x_{eq}$')
ax.set_title(r"Bifurcation diagram: $x' = (x-1)(x-h)$")
from matplotlib.lines import Line2D
handles = [Line2D([0], [0], color='seagreen', lw=2.5, label='stable'),
           Line2D([0], [0], color='crimson', lw=2.5, linestyle='--', label='unstable')]
ax.legend(handles=handles)
plt.tight_layout()
plt.show()

------------------------------------------------------------------------

#### Exercise 1(c): $y' = hy - y^2$

**By Hand.**

Factoring, $f(y) = y(h-y) = 0$ gives equilibria $y=0$ and $y=h$.
Differentiating, $f'(y) = h-2y$, so $$
f'(0) = h, \qquad f'(h) = -h.
$$

- $y=0$ is **stable** for $h<0$ and **unstable** for $h>0$.
- $y=h$ is **unstable** for $h<0$ and **stable** for $h>0$.

At $h=0$ the two equilibria merge at $y=0$ and swap stability — this is
the textbook normal form for a **transcritical bifurcation**.

In [26]:
h_sym, y_sym = sym.symbols('h y')
f = h_sym*y_sym - y_sym**2
eqs = sym.solve(sym.Eq(f,0), y_sym)
fp = sym.diff(f, y_sym)
print("Equilibria:", eqs)
for e in eqs:
    print(f"  f'({e}) =", sym.simplify(fp.subs(y_sym, e)))

Equilibria: [0, h]
  f'(0) = h
  f'(h) = -h

In [27]:
h = np.linspace(-3, 3, 400)
h_lo, h_hi = h[h <= 0], h[h >= 0]

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(h_lo, np.zeros_like(h_lo), color='seagreen', lw=2.5)
ax.plot(h_hi, np.zeros_like(h_hi), color='crimson', lw=2.5, linestyle='--')
ax.plot(h_lo, h_lo, color='crimson', lw=2.5, linestyle='--')
ax.plot(h_hi, h_hi, color='seagreen', lw=2.5)
ax.plot(0, 0, marker='o', mfc='black', mec='black', ms=6)

ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_xlabel('$h$'); ax.set_ylabel('$y_{eq}$')
ax.set_title(r"Bifurcation diagram: $y' = hy - y^2$")
from matplotlib.lines import Line2D
handles = [Line2D([0], [0], color='seagreen', lw=2.5, label='stable'),
           Line2D([0], [0], color='crimson', lw=2.5, linestyle='--', label='unstable')]
ax.legend(handles=handles)
plt.tight_layout()
plt.show()

------------------------------------------------------------------------

#### Exercise 1(d): $x' = (1-x)(x^2-h)$

**By Hand.**

Setting $f(x) = (1-x)(x^2-h) = 0$ gives $x=1$ (for every $h$) together
with $x = \pm\sqrt{h}$ whenever $h \geq 0$ (no real square-root
equilibria when $h<0$). Differentiating, $$
f'(x) = -(x^2-h) + (1-x)(2x) = h + 2x - 3x^2,
$$ so $$
f'(1) = h - 1, \qquad f'(\sqrt{h}) = 2\sqrt{h}\left(1-\sqrt{h}\right), \qquad f'(-\sqrt{h}) = -2\sqrt{h}\left(1+\sqrt{h}\right).
$$

- $x=1$: **stable** for $h<1$, **unstable** for $h>1$.
- $x=-\sqrt{h}$ (exists for $h \geq 0$): since $\sqrt{h}\geq0$,
  $f'(-\sqrt h)\leq 0$ always $\Rightarrow$ **always stable**.
- $x=\sqrt{h}$ (exists for $h \geq 0$):
  $f'(\sqrt h) = 2\sqrt h(1-\sqrt h)$ is positive for $0<h<1$
  (**unstable**) and negative for $h>1$ (**stable**).

Two qualitative changes occur as $h$ varies:

- **At $h=0$:** for $h<0$ there is only the single equilibrium $x=1$; as
  $h$ increases through $0$, a pair of equilibria $x=\pm\sqrt h$ is born
  out of the degenerate double root at $x=0$ (where, at $h=0$,
  $f(x,0)=(1-x)x^2$ is semistable: attracting from the left, repelling
  from the right). This is a **saddle-node bifurcation**.
- **At $h=1$:** the branches $x=1$ and $x=\sqrt h$ meet (both equal $1$
  there) and exchange stability — a **transcritical bifurcation**,
  exactly as in parts (b) and (c).

In [28]:
h_sym, x_sym = sym.symbols('h x')
f = (1-x_sym)*(x_sym**2 - h_sym)
eqs = sym.solve(sym.Eq(f,0), x_sym)
fp = sym.diff(f, x_sym)
print("Equilibria:", eqs)
for e in eqs:
    print(f"  f'({e}) =", sym.simplify(fp.subs(x_sym, e)))

Equilibria: [1, -sqrt(h), sqrt(h)]
  f'(1) = h - 1
  f'(-sqrt(h)) = -2*sqrt(h) - 2*h
  f'(sqrt(h)) = 2*sqrt(h) - 2*h

In [29]:
h_pos = np.linspace(0, 3, 400)
h_all = np.linspace(-3, 3, 400)

fig, ax = plt.subplots(figsize=(6.5, 5.5))

h_lo, h_hi = h_all[h_all <= 1], h_all[h_all >= 1]
ax.plot(h_lo, np.ones_like(h_lo), color='seagreen', lw=2.5)
ax.plot(h_hi, np.ones_like(h_hi), color='crimson', lw=2.5, linestyle='--')

h1, h2 = h_pos[h_pos <= 1], h_pos[h_pos >= 1]
ax.plot(h1, np.sqrt(h1), color='crimson', lw=2.5, linestyle='--')
ax.plot(h2, np.sqrt(h2), color='seagreen', lw=2.5)
ax.plot(h_pos, -np.sqrt(h_pos), color='seagreen', lw=2.5)

ax.plot(0, 0, marker='o', mfc='black', mec='black', ms=6)
ax.plot(1, 1, marker='o', mfc='black', mec='black', ms=6)

ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_xlabel('$h$'); ax.set_ylabel('$x_{eq}$')
ax.set_title(r"Bifurcation diagram: $x' = (1-x)(x^2-h)$", pad=15)
ax.set_ylim(-1.9, 2.3)
from matplotlib.lines import Line2D
handles = [Line2D([0], [0], color='seagreen', lw=2.5, label='stable'),
           Line2D([0], [0], color='crimson', lw=2.5, linestyle='--', label='unstable')]
ax.legend(handles=handles, loc='upper left')
ax.annotate('saddle-node\n(h=0)', xy=(0, 0), xytext=(-2.7, -1.6), fontsize=9,
            arrowprops=dict(arrowstyle='->'))
ax.annotate('transcritical\n(h=1)', xy=(1, 1), xytext=(1.7, 1.75), fontsize=9,
            arrowprops=dict(arrowstyle='->'))
plt.tight_layout()
plt.show()

## Homework 4

### Exercise 1 — Interpreting Numerical Output

Consider the initial value problem $x' = -2tx,\ x(0)=1$, which has exact
solution $x(t) = e^{-t^2}$. Euler’s method was used to approximate
$x(1)$ for several step sizes $h$:

| $h$     | Euler approximation $X(1)$ | Error $\lvert x(1)-X(1)\rvert$ |
|---------|----------------------------|--------------------------------|
| $0.2$   | $0.399383$                 | $3.150\times10^{-2}$           |
| $0.1$   | $0.381707$                 | $1.383\times10^{-2}$           |
| $0.05$  | $0.374384$                 | $6.505\times10^{-3}$           |
| $0.025$ | $0.371036$                 | $3.157\times10^{-3}$           |

#### (a) Ratio of consecutive errors

In [30]:
hs = [0.2, 0.1, 0.05, 0.025]
errs = [3.150e-2, 1.383e-2, 6.505e-3, 3.157e-3]

for i in range(3):
    ratio = errs[i] / errs[i+1]
    print(f"Error({hs[i]}) / Error({hs[i+1]}) = {ratio:.4f}")

Error(0.2) / Error(0.1) = 2.2777
Error(0.1) / Error(0.05) = 2.1261
Error(0.05) / Error(0.025) = 2.0605

Each ratio is close to $2$. This is exactly what we expect from a method
with **global error $O(h)$**: if $\text{Error}(h) \approx Ch$ for some
constant $C$ (independent of $h$, for $h$ small), then halving $h$
should halve the error: $$
\frac{\text{Error}(h)}{\text{Error}(h/2)} \approx \frac{Ch}{C(h/2)} = 2.
$$ The observed ratios ($2.28, 2.13, 2.06$) are close to, and
approaching, $2$ as $h$ shrinks — consistent with the $O(h)$ theory,
with the discrepancy from exactly $2$ attributable to higher-order terms
that become less significant as $h \to 0$.

#### (b) Predicting the error at $h=0.0125$

Since $h=0.0125$ is exactly half of $h=0.025$, and the error should
again roughly halve (per part (a)): $$
\text{Error}(0.0125) \approx \frac{\text{Error}(0.025)}{2} = \frac{3.157\times10^{-3}}{2} \approx 1.58\times10^{-3}.
$$

In [31]:
predicted = errs[3] / 2
print(f"Predicted error at h = 0.0125: {predicted:.3e}")

Predicted error at h = 0.0125: 1.579e-03

> **Tip**
>
> This prediction requires no further computation — it follows directly
> from the observed halving pattern (or equivalently, from the $O(h)$
> global error rate) without running Euler’s method again.

#### (c) Step size needed to reduce the error by a factor of 8

Since Euler’s method is first-order accurate,
$\text{Error}(h) \approx Ch$, so error scales **linearly** with $h$. To
reduce the error by a factor of $8$, we need to shrink $h$ by the same
factor of $8$: $$
\frac{\text{Error}(h)}{\text{Error}(h/8)} = \frac{Ch}{C(h/8)} = 8.
$$

$$
\boxed{\text{Shrink } h \text{ by a factor of } 8 \text{ for Euler's method.}}
$$

For the fourth-order Runge–Kutta (RK4) method, the global error is
$O(h^4)$, i.e. $\text{Error}(h) \approx Ch^4$. To reduce the error by a
factor of $8$, we need $$
\frac{\text{Error}(h)}{\text{Error}(h/r)} = \frac{Ch^4}{C(h/r)^4} = r^4 = 8 \quad \Longrightarrow \quad r = 8^{1/4}.
$$

In [32]:
r = 8**(sym.Rational(1,4))
print("Required factor for RK4:", sym.N(r, 5))

Required factor for RK4: 1.6818

$$
\boxed{\text{Shrink } h \text{ by a factor of } 8^{1/4} \approx 1.68 \text{ for RK4.}}
$$

> **Note**
>
> This makes sense intuitively: because RK4’s error already shrinks much
> faster as $h$ decreases (fourth power vs. first power), a much smaller
> reduction in $h$ is needed to achieve the same reduction in error,
> compared to Euler’s method.

------------------------------------------------------------------------

### Section 2.2.2 — Second-Order Linear Homogeneous Equations

#### Exercise 1: General Solutions

**(a) $x'' - 4x' + 4x = 0$.**

The characteristic equation is
$\lambda^2 - 4\lambda + 4 = (\lambda-2)^2 = 0$, giving a **repeated
root** $\lambda = 2$. For a repeated root, the general solution is $$
\boxed{x(t) = (C_1 + C_2 t)e^{2t}.}
$$

**(b) $x'' - 2x' = 0$.**

The characteristic equation is
$\lambda^2 - 2\lambda = \lambda(\lambda-2) = 0$, giving **distinct real
roots** $\lambda = 0, 2$. $$
\boxed{x(t) = C_1 + C_2 e^{2t}.}
$$

**(c) $\tfrac12 x'' + x' + \tfrac12 x = 0$.**

Multiplying by $2$: $x'' + 2x' + x = 0$, with characteristic equation
$\lambda^2+2\lambda+1=(\lambda+1)^2=0$, a **repeated root**
$\lambda=-1$. $$
\boxed{x(t) = (C_1 + C_2 t)e^{-t}.}
$$

**(d) $x'' + 4x' + 3x = 0$.**

The characteristic equation is
$\lambda^2+4\lambda+3=(\lambda+1)(\lambda+3)=0$, giving **distinct real
roots** $\lambda=-1,-3$. $$
\boxed{x(t) = C_1 e^{-t} + C_2 e^{-3t}.}
$$

In [33]:
t = sym.symbols('t')
x = sym.Function('x')

for label, eq in [
    ("(a)", sym.Eq(sym.diff(x(t),t,2) - 4*sym.diff(x(t),t) + 4*x(t), 0)),
    ("(b)", sym.Eq(sym.diff(x(t),t,2) - 2*sym.diff(x(t),t), 0)),
    ("(c)", sym.Eq(sym.Rational(1,2)*sym.diff(x(t),t,2) + sym.diff(x(t),t) + sym.Rational(1,2)*x(t), 0)),
    ("(d)", sym.Eq(sym.diff(x(t),t,2) + 4*sym.diff(x(t),t) + 3*x(t), 0)),
]:
    print(label, sym.dsolve(eq, x(t)))

(a) Eq(x(t), (C1 + C2*t)*exp(2*t))
(b) Eq(x(t), C1 + C2*exp(2*t))
(c) Eq(x(t), (C1 + C2*t)*exp(-t))
(d) Eq(x(t), (C1 + C2*exp(-2*t))*exp(-t))

> **Note**
>
> SymPy’s general solutions agree with each boxed formula above (note
> $(C_1 + C_2 e^{-2t})e^{-t} = C_1e^{-t} + C_2e^{-3t}$ for part (d),
> just with the constant relabeled).

#### Exercise 2: Initial Value Problems $x(0)=1,\ x'(0)=0$

**(a).** From $x(t) = (C_1+C_2t)e^{2t}$: $x(0)=C_1=1$. Differentiating,
$x'(t) = C_2e^{2t}+2(C_1+C_2t)e^{2t}$, so
$x'(0) = C_2+2C_1 = 0 \Rightarrow C_2=-2$. $$
\boxed{x(t) = (1-2t)e^{2t}.}
$$

**(b).** From $x(t)=C_1+C_2e^{2t}$: $x(0)=C_1+C_2=1$, and
$x'(t)=2C_2e^{2t}$ so $x'(0)=2C_2=0\Rightarrow C_2=0,\ C_1=1$. $$
\boxed{x(t) = 1.}
$$

**(c).** From $x(t)=(C_1+C_2t)e^{-t}$: $x(0)=C_1=1$. Differentiating,
$x'(t)=C_2e^{-t}-(C_1+C_2t)e^{-t}$, so
$x'(0)=C_2-C_1=0\Rightarrow C_2=1$. $$
\boxed{x(t) = (1+t)e^{-t}.}
$$

**(d).** From $x(t)=C_1e^{-t}+C_2e^{-3t}$: $x(0)=C_1+C_2=1$, and
$x'(t)=-C_1e^{-t}-3C_2e^{-3t}$, so
$x'(0)=-C_1-3C_2=0\Rightarrow C_1=-3C_2$. Substituting:
$-3C_2+C_2=1\Rightarrow C_2=-\tfrac12,\ C_1=\tfrac32$. $$
\boxed{x(t) = \frac32 e^{-t} - \frac12 e^{-3t}.}
$$

In [34]:
for label, eq in [
    ("(a)", sym.Eq(sym.diff(x(t),t,2) - 4*sym.diff(x(t),t) + 4*x(t), 0)),
    ("(b)", sym.Eq(sym.diff(x(t),t,2) - 2*sym.diff(x(t),t), 0)),
    ("(c)", sym.Eq(sym.Rational(1,2)*sym.diff(x(t),t,2) + sym.diff(x(t),t) + sym.Rational(1,2)*x(t), 0)),
    ("(d)", sym.Eq(sym.diff(x(t),t,2) + 4*sym.diff(x(t),t) + 3*x(t), 0)),
]:
    sol = sym.dsolve(eq, x(t), ics={x(0): 1, x(t).diff(t).subs(t, 0): 0})
    print(label, sym.simplify(sol))

(a) Eq(x(t), (1 - 2*t)*exp(2*t))
(b) Eq(x(t), 1)
(c) Eq(x(t), (t + 1)*exp(-t))
(d) Eq(x(t), (3*exp(2*t) - 1)*exp(-3*t)/2)

> **Note**
>
> SymPy confirms each particular solution, including the somewhat
> surprising part (b), where the initial conditions collapse the general
> solution to the constant $x(t)=1$.

------------------------------------------------------------------------

### Section 2.2.3 — Second-Order Linear Homogeneous Equations (Complex & Real Roots)

#### Exercise 1: General Solutions

For each part, we find the roots of the characteristic equation and
write the corresponding general solution.

**(a) $x''+x'+4x=0$.**
$\lambda = \dfrac{-1\pm\sqrt{1-16}}{2} = -\dfrac12 \pm \dfrac{\sqrt{15}}{2}i$
(complex). $$
\boxed{x(t) = e^{-t/2}\left(C_1\cos\tfrac{\sqrt{15}}{2}t + C_2\sin\tfrac{\sqrt{15}}{2}t\right).}
$$

**(b) $x''-4x'+6x=0$.**
$\lambda = \dfrac{4\pm\sqrt{16-24}}{2} = 2\pm\sqrt2\,i$ (complex). $$
\boxed{x(t) = e^{2t}\left(C_1\cos(\sqrt2\,t) + C_2\sin(\sqrt2\,t)\right).}
$$

**(c) $x''+9x=0$.** $\lambda = \pm 3i$ (purely imaginary). $$
\boxed{x(t) = C_1\cos(3t) + C_2\sin(3t).}
$$

**(d) $x''-12x=0$.** $\lambda = \pm2\sqrt3$ (distinct real). $$
\boxed{x(t) = C_1e^{2\sqrt3\,t} + C_2e^{-2\sqrt3\,t}.}
$$

**(e) $2x''+3x'+3x=0$.**
$\lambda = \dfrac{-3\pm\sqrt{9-24}}{4} = -\dfrac34 \pm \dfrac{\sqrt{15}}{4}i$
(complex). $$
\boxed{x(t) = e^{-3t/4}\left(C_1\cos\tfrac{\sqrt{15}}{4}t + C_2\sin\tfrac{\sqrt{15}}{4}t\right).}
$$

**(f) $\tfrac12x''+\tfrac56x'+\tfrac29x=0$.** Multiplying by $18$:
$9x''+15x'+4x=0$, so
$\lambda = \dfrac{-15\pm\sqrt{225-144}}{18} = \dfrac{-15\pm9}{18}$,
giving $\lambda = -\tfrac13, -\tfrac43$ (distinct real). $$
\boxed{x(t) = C_1e^{-t/3} + C_2e^{-4t/3}.}
$$

In [35]:
for label, eq in [
    ("(a)", sym.Eq(sym.diff(x(t),t,2) + sym.diff(x(t),t) + 4*x(t), 0)),
    ("(b)", sym.Eq(sym.diff(x(t),t,2) - 4*sym.diff(x(t),t) + 6*x(t), 0)),
    ("(c)", sym.Eq(sym.diff(x(t),t,2) + 9*x(t), 0)),
    ("(d)", sym.Eq(sym.diff(x(t),t,2) - 12*x(t), 0)),
    ("(e)", sym.Eq(2*sym.diff(x(t),t,2) + 3*sym.diff(x(t),t) + 3*x(t), 0)),
    ("(f)", sym.Eq(sym.Rational(1,2)*sym.diff(x(t),t,2) + sym.Rational(5,6)*sym.diff(x(t),t) + sym.Rational(2,9)*x(t), 0)),
]:
    print(label, sym.simplify(sym.dsolve(eq, x(t))))

(a) Eq(x(t), (C1*sin(sqrt(15)*t/2) + C2*cos(sqrt(15)*t/2))*exp(-t/2))
(b) Eq(x(t), (C1*sin(sqrt(2)*t) + C2*cos(sqrt(2)*t))*exp(2*t))
(c) Eq(x(t), C1*sin(3*t) + C2*cos(3*t))
(d) Eq(x(t), (C1 + C2*exp(4*sqrt(3)*t))*exp(-2*sqrt(3)*t))
(e) Eq(x(t), (C1*sin(sqrt(15)*t/4) + C2*cos(sqrt(15)*t/4))*exp(-3*t/4))
(f) Eq(x(t), (C1*exp(t) + C2)*exp(-4*t/3))

> **Note**
>
> All six general solutions match. (Part (d)’s solution can equivalently
> be written as $x(t) = A\cosh(2\sqrt3\,t) + B\sinh(2\sqrt3\,t)$.)

#### Exercise 2: Initial Value Problems $x(0)=1,\ x'(0)=0$

For each complex-root case, writing
$x(t) = e^{\alpha t}(C_1\cos\beta t + C_2\sin\beta t)$, the conditions
$x(0)=1$ and $x'(0)=0$ give $C_1=1$ and $C_2 = -\alpha/\beta$ (from
$x'(0) = \alpha C_1 + \beta C_2 = 0$).

**(a).** $\alpha=-\tfrac12,\ \beta=\tfrac{\sqrt{15}}{2}$, so
$C_2 = \tfrac{1/2}{\sqrt{15}/2} = \tfrac{1}{\sqrt{15}}$. $$
\boxed{x(t) = e^{-t/2}\left(\cos\tfrac{\sqrt{15}}{2}t + \tfrac{1}{\sqrt{15}}\sin\tfrac{\sqrt{15}}{2}t\right).}
$$

**(b).** $\alpha=2,\ \beta=\sqrt2$, so $C_2 = -2/\sqrt2 = -\sqrt2$. $$
\boxed{x(t) = e^{2t}\left(\cos(\sqrt2\,t) - \sqrt2\sin(\sqrt2\,t)\right).}
$$

**(c).** $\alpha=0,\ \beta=3$, so $C_2=0$. $$
\boxed{x(t) = \cos(3t).}
$$

**(d).** $x(t)=C_1e^{2\sqrt3t}+C_2e^{-2\sqrt3t}$: $x(0)=C_1+C_2=1$,
$x'(0)=2\sqrt3(C_1-C_2)=0\Rightarrow C_1=C_2=\tfrac12$. $$
\boxed{x(t) = \tfrac12\left(e^{2\sqrt3t}+e^{-2\sqrt3t}\right) = \cosh(2\sqrt3\,t).}
$$

**(e).** $\alpha=-\tfrac34,\ \beta=\tfrac{\sqrt{15}}{4}$, so
$C_2 = \tfrac{3/4}{\sqrt{15}/4} = \tfrac{3}{\sqrt{15}} = \tfrac{\sqrt{15}}{5}$.
$$
\boxed{x(t) = e^{-3t/4}\left(\cos\tfrac{\sqrt{15}}{4}t + \tfrac{\sqrt{15}}{5}\sin\tfrac{\sqrt{15}}{4}t\right).}
$$

**(f).** $x(t)=C_1e^{-t/3}+C_2e^{-4t/3}$: $x(0)=C_1+C_2=1$,
$x'(0)=-\tfrac13C_1-\tfrac43C_2=0\Rightarrow C_1=-4C_2$. So
$-4C_2+C_2=1\Rightarrow C_2=-\tfrac13,\ C_1=\tfrac43$. $$
\boxed{x(t) = \tfrac43e^{-t/3} - \tfrac13e^{-4t/3}.}
$$

In [36]:
for label, eq in [
    ("(a)", sym.Eq(sym.diff(x(t),t,2) + sym.diff(x(t),t) + 4*x(t), 0)),
    ("(b)", sym.Eq(sym.diff(x(t),t,2) - 4*sym.diff(x(t),t) + 6*x(t), 0)),
    ("(c)", sym.Eq(sym.diff(x(t),t,2) + 9*x(t), 0)),
    ("(d)", sym.Eq(sym.diff(x(t),t,2) - 12*x(t), 0)),
    ("(e)", sym.Eq(2*sym.diff(x(t),t,2) + 3*sym.diff(x(t),t) + 3*x(t), 0)),
    ("(f)", sym.Eq(sym.Rational(1,2)*sym.diff(x(t),t,2) + sym.Rational(5,6)*sym.diff(x(t),t) + sym.Rational(2,9)*x(t), 0)),
]:
    sol = sym.dsolve(eq, x(t), ics={x(0): 1, x(t).diff(t).subs(t, 0): 0})
    print(label, sym.simplify(sol))

(a) Eq(x(t), (sqrt(15)*sin(sqrt(15)*t/2)/15 + cos(sqrt(15)*t/2))*exp(-t/2))
(b) Eq(x(t), (-sqrt(2)*sin(sqrt(2)*t) + cos(sqrt(2)*t))*exp(2*t))
(c) Eq(x(t), cos(3*t))
(d) Eq(x(t), cosh(2*sqrt(3)*t))
(e) Eq(x(t), (sqrt(15)*sin(sqrt(15)*t/4)/5 + cos(sqrt(15)*t/4))*exp(-3*t/4))
(f) Eq(x(t), (4*exp(t) - 1)*exp(-4*t/3)/3)

> **Note**
>
> All six particular solutions match the boxed formulas above.

------------------------------------------------------------------------

### Section 2.2.4 — Damping and Oscillation

#### Exercise 3

For which values of the parameter $k$ (if any) will solutions to
$x''+x'+kx=0$: oscillate with no decay? oscillate with decay? decay
without oscillations?

**By Hand.**

The characteristic equation is $\lambda^2+\lambda+k=0$, with roots $$
\lambda = \frac{-1\pm\sqrt{1-4k}}{2}.
$$

**Case 1: $k > \tfrac14$ (complex roots).** Here $1-4k<0$, so $$
\lambda = -\frac12 \pm \frac{\sqrt{4k-1}}{2}i.
$$ The real part is $-\tfrac12$ for **every** value of $k$ in this range
— it never vanishes. So solutions always decay (since
$\text{Re}(\lambda) = -\tfrac12 < 0$) while oscillating (since the
imaginary part is nonzero). This is **oscillation with decay**, and it
occurs for **every $k > \tfrac14$**.

**Case 2: $0 < k \leq \tfrac14$ (real, negative roots).** Here
$1-4k\ge0$ so both roots are real: $$
\lambda_{1,2} = \frac{-1\pm\sqrt{1-4k}}{2}.
$$ The larger root $\lambda_1 = \dfrac{-1+\sqrt{1-4k}}{2}$ is negative
exactly when $\sqrt{1-4k} < 1$, i.e. when $1-4k<1$, i.e. when $k>0$. So
for $0<k\le\tfrac14$, **both** roots are real and negative, giving
**decay without oscillation** (at $k=\tfrac14$ the roots coincide at
$\lambda=-\tfrac12$, a repeated root, still decaying without
oscillation).

**Case 3: $k \leq 0$.** At $k=0$: roots are $\lambda=0,-1$; the solution
$x(t)=C_1+C_2e^{-t}$ approaches the nonzero constant $C_1$, not decaying
to zero and not oscillating. For $k<0$: $\sqrt{1-4k}>1$, so
$\lambda_1 = \dfrac{-1+\sqrt{1-4k}}{2} > 0$, giving an unstable, growing
(non-decaying) solution.

**Conclusion:**

- **Oscillate with no decay:** *never occurs*, for any real value of
  $k$. Since the coefficient of $x'$ is $1 \neq 0$, the real part of any
  complex root pair is always $-\tfrac12$, which is never zero.
- **Oscillate with decay:** occurs for $\boxed{k > \tfrac14}$.
- **Decay without oscillation:** occurs for
  $\boxed{0 < k \leq \tfrac14}$.

> **Tip**
>
> For $k \le 0$ solutions neither oscillate nor decay to zero (they
> approach a nonzero constant at $k=0$, or grow without bound for $k<0$)
> — worth noting even though the exercise only asks about the three
> listed behaviors.

In [37]:
lam, k = sym.symbols('lambda k', real=True)
char = lam**2 + lam + k
roots = sym.solve(sym.Eq(char, 0), lam)
print("Roots:", roots)
print("Discriminant vanishes (repeated root) at k =", sym.solve(sym.Eq(sym.discriminant(char, lam), 0), k))

for kv in [-0.1, 0, 0.1, 0.2, 0.25, 1]:
    rts = [sym.N(r.subs(k, kv)) for r in sym.solve(sym.Eq(lam**2 + lam + kv, 0), lam)]
    print(f"k={kv}: roots = {rts}")

Roots: [-sqrt(1 - 4*k)/2 - 1/2, sqrt(1 - 4*k)/2 - 1/2]
Discriminant vanishes (repeated root) at k = [1/4]
k=-0.1: roots = [-1.09160797830996, 0.0916079783099616]
k=0: roots = [-1.00000000000000, 0]
k=0.1: roots = [-0.887298334620742, -0.112701665379258]
k=0.2: roots = [-0.723606797749979, -0.276393202250021]
k=0.25: roots = [-0.500000000000000]
k=1: roots = []

> **Note**
>
> The numerical sweep confirms the case analysis: negative real part
> throughout, a repeated real root exactly at $k=1/4$, real roots for
> $k \le 1/4$, and a sign change in the larger real root exactly at
> $k=0$.

------------------------------------------------------------------------

### Section 2.3.1 — Undetermined Coefficients

#### Exercise 1: Form of the Trial Guess

For each source term $f(t)$, state the *form* of an initial trial guess
for a particular solution $x_p(t)$ (before checking for duplication with
the homogeneous solution).

**(a) $f(t) = 3t^3-1$** (a degree-3 polynomial). $$
\boxed{x_p(t) = At^3+Bt^2+Ct+D.}
$$

**(c) $f(t) = 12$** (a constant, i.e. a degree-0 polynomial). $$
\boxed{x_p(t) = A.}
$$

**(e) $f(t) = 5\sin 7t$** (a pure sinusoid of frequency $7$). $$
\boxed{x_p(t) = A\cos(7t) + B\sin(7t).}
$$

> **Tip**
>
> The general rule: match the “family” of $f(t)$ — a degree-$n$
> polynomial pairs with a full degree-$n$ polynomial trial guess; a sine
> or cosine of frequency $\omega$ pairs with *both* $\cos\omega t$ and
> $\sin\omega t$ (even if only one appears in $f$), because
> differentiation mixes the two.

#### Exercise 2: General Solutions

**(a) $x''+7x = te^{3t}$.**

*Homogeneous:* $\lambda^2+7=0 \Rightarrow \lambda=\pm\sqrt7\,i$, so
$x_h = C_1\cos(\sqrt7t)+C_2\sin(\sqrt7t)$.

*Particular:* Since $f(t)=te^{3t}$ and $3$ is not a root of the
characteristic equation, try $x_p = (At+B)e^{3t}$. Substituting and
matching coefficients (worked below with SymPy) gives $A=\tfrac{1}{16}$,
$B=-\tfrac{3}{128}$.

$$
\boxed{x(t) = C_1\cos(\sqrt7t)+C_2\sin(\sqrt7t) + \left(\frac{t}{16}-\frac{3}{128}\right)e^{3t}.}
$$

**(d) $x''-3x'-4x = 2t^2$.**

*Homogeneous:*
$\lambda^2-3\lambda-4=(\lambda-4)(\lambda+1)=0 \Rightarrow \lambda=4,-1$,
so $x_h = C_1e^{4t}+C_2e^{-t}$.

*Particular:* Since $f(t)=2t^2$ and $0$ is not a root, try
$x_p=At^2+Bt+C$. Substituting:
$2A-3(2At+B)-4(At^2+Bt+C) = -4At^2+(-6A-4B)t+(2A-3B-4C)$. Matching to
$2t^2$: $A=-\tfrac12$, then $B=\tfrac34$, then $C=-\tfrac{13}{16}$.

$$
\boxed{x(t) = C_1e^{4t}+C_2e^{-t} - \frac{t^2}{2}+\frac{3t}{4}-\frac{13}{16}.}
$$

**(h) $x''+x'+2x = t\sin2t$.**

*Homogeneous:*
$\lambda^2+\lambda+2=0 \Rightarrow \lambda=-\tfrac12\pm\tfrac{\sqrt7}{2}i$,
so
$x_h = e^{-t/2}\left(C_1\cos\tfrac{\sqrt7}{2}t+C_2\sin\tfrac{\sqrt7}{2}t\right)$.

*Particular:* Since $f(t) = t\sin2t$ and $2i$ is not a root of the
characteristic equation, try $x_p = (At+B)\cos2t+(Ct+D)\sin2t$.
Substituting and matching coefficients (worked below) gives
$A=C=-\tfrac14,\ B=-\tfrac18,\ D=\tfrac12$.

$$
\boxed{x(t) = e^{-t/2}\left(C_1\cos\tfrac{\sqrt7}{2}t+C_2\sin\tfrac{\sqrt7}{2}t\right) + \left(-\frac{t}{4}-\frac18\right)\cos2t + \left(-\frac{t}{4}+\frac12\right)\sin2t.}
$$

In [38]:
t = sym.symbols('t')
x = sym.Function('x')

for label, eq in [
    ("(a)", sym.Eq(sym.diff(x(t),t,2) + 7*x(t), t*sym.exp(3*t))),
    ("(d)", sym.Eq(sym.diff(x(t),t,2) - 3*sym.diff(x(t),t) - 4*x(t), 2*t**2)),
    ("(h)", sym.Eq(sym.diff(x(t),t,2) + sym.diff(x(t),t) + 2*x(t), t*sym.sin(2*t))),
]:
    print(label, sym.simplify(sym.dsolve(eq, x(t))))

(a) Eq(x(t), C1*sin(sqrt(7)*t) + C2*cos(sqrt(7)*t) + t*exp(3*t)/16 - 3*exp(3*t)/128)
(d) Eq(x(t), C1*exp(-t) + C2*exp(4*t) - t**2/2 + 3*t/4 - 13/16)
(h) Eq(x(t), (C1*sin(sqrt(7)*t/2) + C2*cos(sqrt(7)*t/2) + (-2*sqrt(2)*t*sin(2*t + pi/4) + 4*sin(2*t) - cos(2*t))*exp(t/2)/8)*exp(-t/2))

> **Note**
>
> SymPy’s results agree with the boxed formulas (for part (h), SymPy’s
> particular solution is written in a shifted-phase form; expanding
> $\sin(2t+\pi/4)$ and simplifying shows it is algebraically identical
> to the by-hand result).

------------------------------------------------------------------------

### Section 2.4.2 — Variation of Parameters

#### Exercise 1: Particular Solutions via Variation of Parameters

For $x''+p(t)x'+q(t)x=f(t)$ with homogeneous solutions $y_1, y_2$ and
Wronskian $W=y_1y_2'-y_2y_1'$, the variation-of-parameters formula gives
a particular solution $$
x_p(t) = -y_1(t)\int \frac{y_2(t)f(t)}{W(t)}\,dt + y_2(t)\int\frac{y_1(t)f(t)}{W(t)}\,dt.
$$

**(a) $x''+x = \tan t$.**

Homogeneous solutions: $y_1=\cos t,\ y_2=\sin t$, with Wronskian
$W = \cos t\cdot\cos t - \sin t\cdot(-\sin t) = 1$.

$$
x_p = -\cos t\int \sin t\tan t\,dt + \sin t\int\cos t\tan t\,dt.
$$

Since $\cos t\tan t = \sin t$, the second integral is
$\int \sin t\,dt = -\cos t$. Since
$\sin t\tan t = \dfrac{\sin^2t}{\cos t} = \sec t - \cos t$, the first
integral is $\int(\sec t - \cos t)\,dt = \ln|\sec t+\tan t| - \sin t$.
Substituting back: $$
x_p = -\cos t\left(\ln|\sec t+\tan t| - \sin t\right) + \sin t(-\cos t) = -\cos t\ln|\sec t+\tan t| + \sin t\cos t - \sin t\cos t.
$$

The $\sin t\cos t$ terms cancel, leaving

$$
\boxed{x_p(t) = -\cos t\,\ln\left|\sec t+\tan t\right|.}
$$

In [39]:
t = sym.symbols('t', positive=True)

y1, y2 = sym.cos(t), sym.sin(t)
W = y1*sym.diff(y2, t) - y2*sym.diff(y1, t)
f = sym.tan(t)

xp = -y1*sym.integrate(y2*f/W, t) + y2*sym.integrate(y1*f/W, t)
xp_boxed = -sym.cos(t)*sym.log(sym.sec(t) + sym.tan(t))

residual = sym.simplify(sym.diff(xp_boxed, t, 2) + xp_boxed - f)
print("Residual for boxed x_p (should be 0):", residual)

Residual for boxed x_p (should be 0): 0

> **Note**
>
> Direct substitution confirms $x_p(t) = -\cos t\ln|\sec t+\tan t|$
> satisfies $x_p''+x_p = \tan t$ exactly.

**(c) $x''-x = \dfrac1t$.**

Homogeneous solutions: $y_1=e^t,\ y_2=e^{-t}$, with Wronskian
$W = e^t(-e^{-t}) - e^{-t}e^t = -2$.

$$
x_p = -e^t\int e^{-t}\cdot\frac{1}{-2t}\,dt + e^{-t}\int e^t\cdot\frac{1}{-2t}\,dt = \frac12 e^t\int\frac{e^{-t}}{t}\,dt - \frac12 e^{-t}\int\frac{e^t}{t}\,dt.
$$

Neither integral is elementary — they are (up to sign/branch
conventions) the **exponential integral function** $\mathrm{Ei}$: $$
\int\frac{e^t}{t}\,dt = \mathrm{Ei}(t)+C, \qquad \int\frac{e^{-t}}{t}\,dt = \mathrm{Ei}(-t)+C \quad (t>0).
$$

$$
\boxed{x_p(t) = \frac12 e^t\,\mathrm{Ei}(-t) - \frac12 e^{-t}\,\mathrm{Ei}(t).}
$$

In [40]:
from scipy.special import expi

# Numerically verify x_p'' - x_p = 1/t at several points
def xp_c(t0, h=1e-5):
    def f(tt):
        return 0.5*np.exp(tt)*expi(-tt) - 0.5*np.exp(-tt)*expi(tt)
    d2 = (f(t0 + h) - 2*f(t0) + f(t0 - h)) / h**2
    return d2 - f(t0)

for t0 in [0.5, 1.0, 2.0, 3.0]:
    print(f"t={t0}: x_p'' - x_p = {xp_c(t0):.6f}, expected 1/t = {1/t0:.6f}")

t=0.5: x_p'' - x_p = 2.000003, expected 1/t = 2.000000
t=1.0: x_p'' - x_p = 1.000007, expected 1/t = 1.000000
t=2.0: x_p'' - x_p = 0.500002, expected 1/t = 0.500000
t=3.0: x_p'' - x_p = 0.333333, expected 1/t = 0.333333

> **Note**
>
> The residual $x_p''-x_p$ matches $1/t$ closely at every test point
> (small differences are finite-difference truncation error, not a flaw
> in the formula).

**(e) $x''+x = \dfrac{1}{t+1}$.**

Homogeneous solutions and Wronskian are the same as in part (a):
$y_1=\cos t,\ y_2=\sin t,\ W=1$. Substituting $u=t+1$ (so $t=u-1$) and
using the angle-subtraction formulas, the integrals become the **sine
and cosine integral functions** $\mathrm{Si}, \mathrm{Ci}$: $$
\int\frac{\sin t}{t+1}\,dt = \cos(1)\,\mathrm{Si}(t+1) - \sin(1)\,\mathrm{Ci}(t+1), \qquad
\int\frac{\cos t}{t+1}\,dt = \cos(1)\,\mathrm{Ci}(t+1) + \sin(1)\,\mathrm{Si}(t+1).
$$

Therefore

$$
\boxed{x_p(t) = -\cos t\Big[\cos(1)\mathrm{Si}(t+1) - \sin(1)\mathrm{Ci}(t+1)\Big] + \sin t\Big[\cos(1)\mathrm{Ci}(t+1) + \sin(1)\mathrm{Si}(t+1)\Big].}
$$

In [41]:
from scipy.special import sici

c1, s1 = np.cos(1), np.sin(1)

def xp_e(t0, h=1e-5):
    def f(tt):
        Si_u, Ci_u = sici(tt + 1)
        return -np.cos(tt)*(c1*Si_u - s1*Ci_u) + np.sin(tt)*(c1*Ci_u + s1*Si_u)
    d2 = (f(t0 + h) - 2*f(t0) + f(t0 - h)) / h**2
    return d2 + f(t0)

for t0 in [0.0, 0.5, 1.0, 2.0]:
    print(f"t={t0}: x_p'' + x_p = {xp_e(t0):.6f}, expected 1/(t+1) = {1/(t0+1):.6f}")

t=0.0: x_p'' + x_p = 1.000000, expected 1/(t+1) = 1.000000
t=0.5: x_p'' + x_p = 0.666663, expected 1/(t+1) = 0.666667
t=1.0: x_p'' + x_p = 0.500003, expected 1/(t+1) = 0.500000
t=2.0: x_p'' + x_p = 0.333332, expected 1/(t+1) = 0.333333

> **Note**
>
> Again, direct numerical substitution confirms the closed-form
> particular solution — a good check when an integral doesn’t reduce to
> elementary functions.

> **Tip**
>
> Parts (c) and (e) illustrate an important point: the variation of
> parameters formula *always* produces a valid particular solution in
> principle, but the resulting integrals are not always expressible in
> terms of elementary functions. When that happens, it’s standard
> practice to express the answer using well-studied special functions
> (here, $\mathrm{Ei}$, $\mathrm{Si}$, $\mathrm{Ci}$) rather than
> leaving the answer as an unevaluated integral.

## Homework 5

### Section 3.1

#### Exercise 6

Use the definition of the Laplace transform to show that $$
\mathcal{L}[f(t)H(t-a)] = e^{-as}\mathcal{L}[f(t+a)].
$$

**By Hand.**

By definition, $$
\mathcal{L}[f(t)H(t-a)] = \int_0^\infty f(t)H(t-a)e^{-st}\,dt.
$$

Since $H(t-a) = 0$ for $t<a$ and $H(t-a)=1$ for $t>a$, the integrand
vanishes on $[0,a)$, so $$
\mathcal{L}[f(t)H(t-a)] = \int_a^\infty f(t)e^{-st}\,dt.
$$

Now substitute $\tau = t-a$ (so $t = \tau+a$, $dt=d\tau$); when $t=a$,
$\tau=0$, and as $t\to\infty$, $\tau\to\infty$: $$
\int_a^\infty f(t)e^{-st}\,dt = \int_0^\infty f(\tau+a)\,e^{-s(\tau+a)}\,d\tau = e^{-as}\int_0^\infty f(\tau+a)\,e^{-s\tau}\,d\tau.
$$

The remaining integral is exactly $\mathcal{L}[f(t+a)]$ (with the dummy
variable renamed from $\tau$ back to $t$). Therefore

$$
\boxed{\mathcal{L}[f(t)H(t-a)] = e^{-as}\mathcal{L}[f(t+a)].}
$$

> **Tip**
>
> This is subtly different from the more familiar (second) shift theorem
> $\mathcal{L}[f(t-a)H(t-a)] = e^{-as}F(s)$. Here the function $f$
> multiplying $H(t-a)$ is **not** pre-shifted — it’s the *original*
> $f(t)$, not $f(t-a)$ — so the shift has to be built in by evaluating
> $f$ at $t+a$ inside the transform on the right-hand side.

------------------------------------------------------------------------

#### Exercise 7

Use the preceding exercise to compute $\mathcal{L}[t^2H(t-1)]$.

**By Hand.**

Apply the result of Exercise 6 with $f(t)=t^2$ and $a=1$: $$
\mathcal{L}[t^2H(t-1)] = e^{-s}\mathcal{L}[(t+1)^2] = e^{-s}\mathcal{L}[t^2+2t+1].
$$

Using $\mathcal{L}[t^2]=2/s^3$, $\mathcal{L}[t]=1/s^2$,
$\mathcal{L}[1]=1/s$: $$
\mathcal{L}[(t+1)^2] = \frac{2}{s^3}+\frac{2}{s^2}+\frac{1}{s}.
$$

$$
\boxed{\mathcal{L}[t^2H(t-1)] = e^{-s}\left(\frac{2}{s^3}+\frac{2}{s^2}+\frac{1}{s}\right).}
$$

In [42]:
t, s = sym.symbols('t s', positive=True)

f_shifted = (t+1)**2
F_shifted = sym.laplace_transform(f_shifted, t, s, noconds=True)
print("L[(t+1)^2] =", sym.expand(F_shifted))
print("so L[t^2 H(t-1)] = e^(-s) * (", sym.expand(F_shifted), ")")

# Direct check using Heaviside
H = sym.Heaviside
direct = sym.laplace_transform(t**2*H(t-1), t, s, noconds=True)
mine = sym.exp(-s)*(2/s**3+2/s**2+1/s)
print("Direct L[t^2 H(t-1)] - boxed formula simplifies to:", sym.simplify(direct - mine))

L[(t+1)^2] = 1/s + 2/s**2 + 2/s**3
so L[t^2 H(t-1)] = e^(-s) * ( 1/s + 2/s**2 + 2/s**3 )
Direct L[t^2 H(t-1)] - boxed formula simplifies to: 0

> **Note**
>
> SymPy’s direct computation of $\mathcal{L}[t^2H(t-1)]$ (via the
> Heaviside function) matches the boxed formula exactly.

------------------------------------------------------------------------

#### Exercise 8: Laplace Transforms

**(a) $6+5e^{-2t}+te^{3t}$.**

Using linearity together with $\mathcal{L}[1]=1/s$,
$\mathcal{L}[e^{at}]=1/(s-a)$, and $\mathcal{L}[te^{at}]=1/(s-a)^2$: $$
\boxed{\mathcal{L}\left[6+5e^{-2t}+te^{3t}\right] = \frac{6}{s} + \frac{5}{s+2} + \frac{1}{(s-3)^2}.}
$$

**(b) $tH(t-3)$.**

Apply Exercise 6 with $f(t)=t,\ a=3$:
$\mathcal{L}[tH(t-3)] = e^{-3s}\mathcal{L}[t+3] = e^{-3s}\left(\dfrac{1}{s^2}+\dfrac{3}{s}\right)$.

$$
\boxed{\mathcal{L}[tH(t-3)] = e^{-3s}\left(\frac{1}{s^2}+\frac{3}{s}\right).}
$$

**(c) $\cos 5t$.**

Using $\mathcal{L}[\cos kt] = s/(s^2+k^2)$ with $k=5$:

$$
\boxed{\mathcal{L}[\cos5t] = \frac{s}{s^2+25}.}
$$

**(f) $H(t-\pi)\cos(t-\pi)$.**

This is already in the shifted form $f(t-a)H(t-a)$ with
$f(t)=\cos t,\ a=\pi$. By the (second) shift theorem,
$\mathcal{L}[f(t-a)H(t-a)] = e^{-as}F(s)$ where
$F(s) = \mathcal{L}[\cos t] = s/(s^2+1)$.

$$
\boxed{\mathcal{L}[H(t-\pi)\cos(t-\pi)] = \frac{s\,e^{-\pi s}}{s^2+1}.}
$$

In [43]:
H = sym.Heaviside

for label, f in [
    ("(a)", 6 + 5*sym.exp(-2*t) + t*sym.exp(3*t)),
    ("(b)", t*H(t-3)),
    ("(c)", sym.cos(5*t)),
    ("(f)", H(t-sym.pi)*sym.cos(t-sym.pi)),
]:
    F = sym.laplace_transform(f, t, s, noconds=True)
    print(label, sym.simplify(F))

(a) 5/(s + 2) + (s - 3)**(-2) + 6/s
(b) (3*s + 1)*exp(-3*s)/s**2
(c) s/(s**2 + 25)
(f) s*exp(-pi*s)/(s**2 + 1)

> **Note**
>
> All four SymPy results agree with the boxed formulas above.

------------------------------------------------------------------------

#### Exercise 9: Inverse Laplace Transforms

**(c) $\dfrac{2}{(s-5)^2}$.**

Using $\mathcal{L}^{-1}\left[\dfrac{1}{(s-a)^2}\right] = te^{at}$ with
$a=5$:

$$
\boxed{\mathcal{L}^{-1}\left[\frac{2}{(s-5)^2}\right] = 2t\,e^{5t}.}
$$

**(d) $\dfrac{7}{s}e^{-4s}$.**

By the (second) shift theorem,
$\mathcal{L}^{-1}[e^{-as}F(s)] = f(t-a)H(t-a)$. Here $F(s)=7/s$
corresponds to $f(t)=7$, and $a=4$:

$$
\boxed{\mathcal{L}^{-1}\left[\frac{7}{s}e^{-4s}\right] = 7H(t-4).}
$$

**(f) $\dfrac{7s+1}{s^2+4}$.**

Split into two familiar pieces: $$
\frac{7s+1}{s^2+4} = \frac{7s}{s^2+4} + \frac{1}{s^2+4} = 7\cdot\frac{s}{s^2+4} + \frac12\cdot\frac{2}{s^2+4}.
$$

Using $\mathcal{L}^{-1}\left[\dfrac{s}{s^2+k^2}\right]=\cos kt$ and
$\mathcal{L}^{-1}\left[\dfrac{k}{s^2+k^2}\right]=\sin kt$ with $k=2$:

$$
\boxed{\mathcal{L}^{-1}\left[\frac{7s+1}{s^2+4}\right] = 7\cos2t + \frac12\sin2t.}
$$

In [44]:
for label, F in [
    ("(c)", 2/(s-5)**2),
    ("(d)", (sym.Rational(7)/s)*sym.exp(-4*s)),
    ("(f)", (7*s+1)/(s**2+4)),
]:
    f = sym.inverse_laplace_transform(F, s, t)
    print(label, sym.simplify(f))

(c) 2*t*exp(5*t)
(d) 7*Heaviside(t - 4)
(f) sin(2*t)/2 + 7*cos(2*t)

> **Note**
>
> All three inverse transforms match the boxed formulas above.

------------------------------------------------------------------------

#### Exercise 12

Use the switching property to find the Laplace transform of $$
x(t) = \begin{cases} 0, & t<2 \\ e^{-t}, & t>2. \end{cases}
$$

**By Hand.**

We can write $x(t) = e^{-t}H(t-2)$. To use the (second) shift theorem
$\mathcal{L}[f(t-a)H(t-a)]=e^{-as}F(s)$, we need to express $e^{-t}$ in
terms of $(t-2)$. Writing $$
e^{-t} = e^{-(t-2)-2} = e^{-2}\,e^{-(t-2)},
$$ we get $x(t) = e^{-2}\,e^{-(t-2)}H(t-2) = e^{-2}\,g(t-2)H(t-2)$ where
$g(t)=e^{-t}$. Then, with $G(s)=\mathcal{L}[g(t)]=\dfrac{1}{s+1}$ and
$a=2$: $$
\mathcal{L}[x(t)] = e^{-2}\cdot e^{-2s}\,G(s) = e^{-2-2s}\cdot\frac{1}{s+1}.
$$

$$
\boxed{\mathcal{L}[x(t)] = \frac{e^{-2(s+1)}}{s+1}.}
$$

In [45]:
H = sym.Heaviside
x = sym.exp(-t)*H(t-2)
X = sym.laplace_transform(x, t, s, noconds=True)
print("L[x(t)] =", sym.simplify(X))

mine = sym.exp(-2*(s+1))/(s+1)
print("Boxed formula:", mine, " — difference:", sym.simplify(X - mine))

L[x(t)] = exp(-2*s - 2)/(s + 1)
Boxed formula: exp(-2*s - 2)/(s + 1)  — difference: 0

> **Note**
>
> SymPy confirms $\mathcal{L}[x(t)] = \dfrac{e^{-2(s+1)}}{s+1}$,
> matching the boxed formula exactly.

------------------------------------------------------------------------

### Section 3.2

#### Exercise 2: Laplace Transforms

**(a) $H(t-\pi)\cos t$.**

Unlike Exercise 8(f) above, here the cosine is **not** pre-shifted —
it’s $\cos t$, not $\cos(t-\pi)$ — so we need the Exercise 6 formula
rather than the ordinary shift theorem. With $f(t)=\cos t,\ a=\pi$: $$
\mathcal{L}[H(t-\pi)\cos t] = e^{-\pi s}\mathcal{L}[\cos(t+\pi)].
$$

Since $\cos(t+\pi) = -\cos t$ (a phase shift of $\pi$ flips the sign),
$\mathcal{L}[\cos(t+\pi)] = -\dfrac{s}{s^2+1}$.

$$
\boxed{\mathcal{L}[H(t-\pi)\cos t] = -\frac{s\,e^{-\pi s}}{s^2+1}.}
$$

> **Tip**
>
> Compare this with Exercise 8(f): the two problems look superficially
> similar but differ by exactly a sign, precisely because
> $\cos(t-\pi) = -\cos t$ — the argument shift there is already “baked
> in,” while here it has to be introduced via the Exercise 6 formula.

**(b) $3t^4$.**

Using $\mathcal{L}[t^n] = n!/s^{n+1}$ with $n=4$:
$\mathcal{L}[t^4] = 4!/s^5 = 24/s^5$.

$$
\boxed{\mathcal{L}[3t^4] = \frac{72}{s^5}.}
$$

**(c) $t^2\sin t$.**

Using the derivative-of-transform property
$\mathcal{L}[t^nf(t)] = (-1)^nF^{(n)}(s)$ with $n=2$ and
$F(s)=\mathcal{L}[\sin t] = \dfrac{1}{s^2+1}$: $$
\mathcal{L}[t^2\sin t] = F''(s).
$$

Differentiating twice: $$
F'(s) = -\frac{2s}{(s^2+1)^2}, \qquad F''(s) = \frac{-2(s^2+1)^2 + 2s\cdot2(s^2+1)(2s)}{(s^2+1)^4} = \frac{6s^2-2}{(s^2+1)^3}.
$$

$$
\boxed{\mathcal{L}[t^2\sin t] = \frac{2(3s^2-1)}{(s^2+1)^3}.}
$$

**(e) $e^{-6t}\sin3t$.**

Using the (first) shift theorem $\mathcal{L}[e^{at}f(t)] = F(s-a)$ with
$F(s)=\mathcal{L}[\sin3t] = \dfrac{3}{s^2+9}$ and $a=-6$ (so
$s\to s-(-6)=s+6$):

$$
\boxed{\mathcal{L}[e^{-6t}\sin3t] = \frac{3}{(s+6)^2+9}.}
$$

In [46]:
H = sym.Heaviside

for label, f in [
    ("(a)", H(t-sym.pi)*sym.cos(t)),
    ("(b)", 3*t**4),
    ("(c)", t**2*sym.sin(t)),
    ("(e)", sym.exp(-6*t)*sym.sin(3*t)),
]:
    F = sym.laplace_transform(f, t, s, noconds=True)
    print(label, sym.simplify(F))

(a) -s*exp(-pi*s)/(s**2 + 1)
(b) 72/s**5
(c) I/(s + I)**3 - I/(s - I)**3
(e) 3/((s + 6)**2 + 9)

> **Note**
>
> SymPy confirms all four transforms, including the sign in part (a) and
> the polynomial-over-cubic form in part (c).

#### Exercise 3: Inverse Laplace Transforms

**(a) $\dfrac{s}{s^2+7s-8}$.**

Factor the denominator: $s^2+7s-8 = (s-1)(s+8)$. Partial fractions: $$
\frac{s}{(s-1)(s+8)} = \frac{A}{s-1}+\frac{B}{s+8}.
$$

Clearing denominators, $s = A(s+8)+B(s-1)$. Setting $s=1$:
$1=9A\Rightarrow A=\tfrac19$. Setting $s=-8$:
$-8=-9B\Rightarrow B=\tfrac89$.

$$
\boxed{\mathcal{L}^{-1}\left[\frac{s}{s^2+7s-8}\right] = \frac19e^{t} + \frac89e^{-8t}.}
$$

**(b) $\dfrac{3-2s}{s^2+2s+10}$.**

Complete the square: $s^2+2s+10 = (s+1)^2+9$. Write the numerator in
terms of $(s+1)$: $$
3-2s = -2(s+1)+5.
$$ So $$
\frac{3-2s}{(s+1)^2+9} = -2\cdot\frac{s+1}{(s+1)^2+9} + 5\cdot\frac{1}{(s+1)^2+9} = -2\cdot\frac{s+1}{(s+1)^2+9} + \frac53\cdot\frac{3}{(s+1)^2+9}.
$$

Using the shift theorem with
$\mathcal{L}^{-1}\left[\dfrac{s}{s^2+9}\right]=\cos3t$ and
$\mathcal{L}^{-1}\left[\dfrac{3}{s^2+9}\right]=\sin3t$:

$$
\boxed{\mathcal{L}^{-1}\left[\frac{3-2s}{s^2+2s+10}\right] = e^{-t}\left(-2\cos3t + \frac53\sin3t\right).}
$$

**(c) $\dfrac{2}{(s-5)^4}$.**

Using
$\mathcal{L}^{-1}\left[\dfrac{1}{(s-a)^{n+1}}\right] = \dfrac{t^ne^{at}}{n!}$
with $a=5,\ n=3$: $$
\mathcal{L}^{-1}\left[\frac{1}{(s-5)^4}\right] = \frac{t^3e^{5t}}{3!} = \frac{t^3e^{5t}}{6}.
$$

$$
\boxed{\mathcal{L}^{-1}\left[\frac{2}{(s-5)^4}\right] = \frac{t^3e^{5t}}{3}.}
$$

**(d) $\dfrac{1}{s(s-2)}e^{-s}$.**

First invert $F(s) = \dfrac{1}{s(s-2)}$ without the exponential factor.
Partial fractions: $$
\frac{1}{s(s-2)} = \frac{A}{s}+\frac{B}{s-2}.
$$ Clearing denominators: $1=A(s-2)+Bs$. At $s=0$: $A=-\tfrac12$. At
$s=2$: $B=\tfrac12$. So $$
f(t) = \mathcal{L}^{-1}[F(s)] = -\frac12 + \frac12e^{2t}.
$$

By the (second) shift theorem,
$\mathcal{L}^{-1}[e^{-s}F(s)] = f(t-1)H(t-1)$:

$$
\boxed{\mathcal{L}^{-1}\left[\frac{e^{-s}}{s(s-2)}\right] = \left(-\frac12+\frac12e^{2(t-1)}\right)H(t-1).}
$$

**(e) $\dfrac{7s+1}{s^2+4}$.**

This is the same expression as Exercise 9(f) in §3.1:

$$
\boxed{\mathcal{L}^{-1}\left[\frac{7s+1}{s^2+4}\right] = 7\cos2t + \frac12\sin2t.}
$$

**(g) $\dfrac{4}{(s-3)^9}$.**

Using
$\mathcal{L}^{-1}\left[\dfrac{1}{(s-a)^{n+1}}\right] = \dfrac{t^ne^{at}}{n!}$
with $a=3,\ n=8$: $$
\mathcal{L}^{-1}\left[\frac{1}{(s-3)^9}\right] = \frac{t^8e^{3t}}{8!} = \frac{t^8e^{3t}}{40320}.
$$

$$
\boxed{\mathcal{L}^{-1}\left[\frac{4}{(s-3)^9}\right] = \frac{t^8e^{3t}}{10080}.}
$$

In [47]:
for label, F in [
    ("(a)", s/(s**2+7*s-8)),
    ("(b)", (3-2*s)/(s**2+2*s+10)),
    ("(c)", 2/(s-5)**4),
    ("(d)", sym.exp(-s)/(s*(s-2))),
    ("(e)", (7*s+1)/(s**2+4)),
    ("(g)", 4/(s-3)**9),
]:
    f = sym.inverse_laplace_transform(F, s, t)
    print(label, sym.simplify(f))

(a) (exp(9*t) + 8)*exp(-8*t)/9
(b) (5*sin(3*t) - 6*cos(3*t))*exp(-t)/3
(c) t**3*exp(5*t)/3
(d) (exp(2*t - 2) - 1)*Heaviside(t - 1)/2
(e) sin(2*t)/2 + 7*cos(2*t)
(g) t**8*exp(3*t)/10080

> **Note**
>
> All six inverse transforms match the boxed formulas above (SymPy’s
> part (d) output, written with `Heaviside(t-1)` factored out, is
> algebraically identical to
> $\left(-\tfrac12+\tfrac12e^{2(t-1)}\right)H(t-1)$).

------------------------------------------------------------------------

## Homework 6

### Section 3.2

#### Exercise 6: Solving IVPs with Laplace Transforms

**(a) $x'+5x=H(t-2),\ x(0)=1$.**

Taking the Laplace transform of both sides, using
$\mathcal{L}[x'] = sX(s)-x(0)$ and $\mathcal{L}[H(t-2)] = e^{-2s}/s$: $$
sX(s) - 1 + 5X(s) = \frac{e^{-2s}}{s} \quad \Longrightarrow \quad X(s)(s+5) = 1 + \frac{e^{-2s}}{s}.
$$

$$
X(s) = \frac{1}{s+5} + \frac{e^{-2s}}{s(s+5)}.
$$

For the second term, first invert $\dfrac{1}{s(s+5)}$ by partial
fractions: $$
\frac{1}{s(s+5)} = \frac{1}{5}\left(\frac{1}{s}-\frac{1}{s+5}\right) \quad \Longrightarrow \quad \mathcal{L}^{-1}\left[\frac{1}{s(s+5)}\right] = \frac15\left(1-e^{-5t}\right).
$$

By the shift theorem,
$\mathcal{L}^{-1}\left[e^{-2s}\cdot\dfrac{1}{s(s+5)}\right] = \dfrac15\left(1-e^{-5(t-2)}\right)H(t-2)$.

$$
\boxed{x(t) = e^{-5t} + \frac15\left(1-e^{-5(t-2)}\right)H(t-2).}
$$

**(b) $x'+x=\sin2t,\ x(0)=0$.**

$$
sX(s)+X(s) = \frac{2}{s^2+4} \quad \Longrightarrow \quad X(s) = \frac{2}{(s+1)(s^2+4)}.
$$

Partial fractions:
$\dfrac{2}{(s+1)(s^2+4)} = \dfrac{A}{s+1}+\dfrac{Bs+C}{s^2+4}$. Clearing
denominators, $2 = A(s^2+4)+(Bs+C)(s+1)$. Setting $s=-1$:
$2=5A\Rightarrow A=\tfrac25$. Matching the $s^2$ coefficient:
$A+B=0\Rightarrow B=-\tfrac25$. Matching the constant:
$4A+C=2\Rightarrow C=\tfrac25$.

$$
X(s) = \frac{2/5}{s+1} + \frac{2}{5}\left(\frac{1}{s^2+4} - \frac{s}{s^2+4}\right).
$$

$$
\boxed{x(t) = \frac25e^{-t} - \frac25\cos2t + \frac15\sin2t.}
$$

**(e) $x''-2x'+2x=e^{-t},\ x(0)=0,\ x'(0)=1$.**

$$
s^2X(s)-1-2sX(s)+2X(s) = \frac{1}{s+1} \quad \Longrightarrow \quad (s^2-2s+2)X(s) = 1+\frac{1}{s+1} = \frac{s+2}{s+1}.
$$

$$
X(s) = \frac{s+2}{(s+1)(s^2-2s+2)} = \frac{s+2}{(s+1)\left[(s-1)^2+1\right]}.
$$

Partial fractions:
$\dfrac{s+2}{(s+1)[(s-1)^2+1]} = \dfrac{A}{s+1} + \dfrac{B(s-1)+C}{(s-1)^2+1}$.
Setting $s=-1$: $1=5A\Rightarrow A=\tfrac15$. Matching coefficients of
the expanded numerator gives $B=-\tfrac15,\ C=\tfrac75$.

$$
X(s) = \frac{1/5}{s+1} - \frac15\cdot\frac{s-1}{(s-1)^2+1} + \frac75\cdot\frac{1}{(s-1)^2+1}.
$$

Using the shift theorem with
$\mathcal{L}^{-1}\left[\dfrac{s}{s^2+1}\right]=\cos t$ and
$\mathcal{L}^{-1}\left[\dfrac{1}{s^2+1}\right]=\sin t$:

$$
\boxed{x(t) = \frac15e^{-t} + e^{t}\left(-\frac15\cos t + \frac75\sin t\right).}
$$

**(j) $x'=2x+H(t-1),\ x(0)=0$.**

$$
sX(s) - 2X(s) = \frac{e^{-s}}{s} \quad \Longrightarrow \quad X(s) = \frac{e^{-s}}{s(s-2)}.
$$

From Homework 5 (§3.2 Exercise 3d),
$\mathcal{L}^{-1}\left[\dfrac{1}{s(s-2)}\right] = -\dfrac12+\dfrac12e^{2t}$.
By the shift theorem:

$$
\boxed{x(t) = \left(-\frac12+\frac12e^{2(t-1)}\right)H(t-1).}
$$

In [48]:
t, s = sym.symbols('t s', positive=True)
x = sym.Function('x')
H = sym.Heaviside

for label, eq, ics in [
    ("(a)", sym.Eq(sym.diff(x(t),t) + 5*x(t), H(t-2)), {x(0): 1}),
    ("(b)", sym.Eq(sym.diff(x(t),t) + x(t), sym.sin(2*t)), {x(0): 0}),
    ("(e)", sym.Eq(sym.diff(x(t),t,2) - 2*sym.diff(x(t),t) + 2*x(t), sym.exp(-t)),
            {x(0): 0, x(t).diff(t).subs(t, 0): 1}),
    ("(j)", sym.Eq(sym.diff(x(t),t), 2*x(t) + H(t-1)), {x(0): 0}),
]:
    sol = sym.dsolve(eq, x(t), ics=ics)
    print(label, sym.simplify(sol))

(a) Eq(x(t), (exp(5*t)*Heaviside(t - 2) - exp(10)*Heaviside(t - 2) + 5)*exp(-5*t)/5)
(b) Eq(x(t), ((sin(2*t) - 2*cos(2*t))*exp(t) + 2)*exp(-t)/5)
(e) Eq(x(t), ((7*sin(t) - cos(t))*exp(2*t) + 1)*exp(-t)/5)
(j) Eq(x(t), (exp(2*t - 2) - 1)*Heaviside(t - 1)/2)

> **Note**
>
> All four of SymPy’s solutions agree with the boxed formulas above
> (allowing for algebraic rearrangement — e.g. SymPy’s
> $e^{10-5t} = e^{-5(t-2)}$ in part (a)).

------------------------------------------------------------------------

#### Exercise 7

Sketch the function $f(t) = 2H(t-3) - 2H(t-4)$ and find its Laplace
transform.

**By Hand.**

Since $H(t-3)=1$ for $t\ge3$ and $H(t-4)=1$ for $t\ge4$, $$
f(t) = \begin{cases} 0, & t<3 \\ 2, & 3\le t<4 \\ 0, & t\ge4, \end{cases}
$$ a rectangular pulse of height $2$ on the interval $[3,4)$.

In [49]:
def f_pulse(tt):
    return np.where((tt >= 3) & (tt < 4), 2.0, 0.0)

t_vals = np.linspace(0, 6, 1000)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t_vals, f_pulse(t_vals), color='steelblue', lw=2)
ax.plot(3, 2, marker='o', mfc='steelblue', mec='steelblue', ms=6, zorder=5)
ax.plot(3, 0, marker='o', mfc='white', mec='steelblue', ms=6, zorder=5)
ax.plot(4, 2, marker='o', mfc='white', mec='steelblue', ms=6, zorder=5)
ax.plot(4, 0, marker='o', mfc='steelblue', mec='steelblue', ms=6, zorder=5)
ax.set_xlabel('$t$')
ax.set_ylabel('$f(t)$')
ax.set_title(r"$f(t) = 2H(t-3) - 2H(t-4)$")
ax.set_ylim(-0.3, 2.5)
plt.tight_layout()
plt.show()

Using $\mathcal{L}[H(t-a)] = e^{-as}/s$:

$$
\boxed{\mathcal{L}[f(t)] = \frac{2e^{-3s}}{s} - \frac{2e^{-4s}}{s} = \frac{2}{s}\left(e^{-3s}-e^{-4s}\right).}
$$

In [50]:
f = 2*H(t-3) - 2*H(t-4)
F = sym.laplace_transform(f, t, s, noconds=True)
print("L[f(t)] =", sym.simplify(F))

mine = (2/s)*(sym.exp(-3*s) - sym.exp(-4*s))
print("Boxed formula simplified:", sym.simplify(mine))
print("Difference:", sym.simplify(F - mine))

L[f(t)] = (2*exp(s) - 2)*exp(-4*s)/s
Boxed formula simplified: (2*exp(s) - 2)*exp(-4*s)/s
Difference: 0

> **Note**
>
> SymPy confirms the transform exactly.

------------------------------------------------------------------------

#### Exercise 11

Solve the initial value problem $$
x'' + 4x = \begin{cases}\cos2t, & 0\le t\le2\pi \\ 0, & t>2\pi,\end{cases} \qquad x(0)=x'(0)=0,
$$ and sketch the solution.

**By Hand.**

**Step 1 — Write the forcing in shifted form.**

The forcing $f(t)$ “switches off” at $t=2\pi$, so write it as $$
f(t) = \cos2t - \cos2t\cdot H(t-2\pi).
$$

Because $\cos2t$ has period $\pi$ (so, in particular, period $2\pi$ as
well), $\cos\big(2(t-2\pi)\big) = \cos(2t-4\pi) = \cos2t$. This lets us
rewrite the second term in genuinely shifted form: $$
f(t) = \cos2t - \cos\big(2(t-2\pi)\big)H(t-2\pi) = g(t) - g(t-2\pi)H(t-2\pi), \qquad g(t)=\cos2t.
$$

**Step 2 — Laplace transform.**

With $G(s) = \mathcal{L}[\cos2t] = \dfrac{s}{s^2+4}$, the shift theorem
gives $\mathcal{L}[f(t)] = G(s)\left(1-e^{-2\pi s}\right)$. Since
$x(0)=x'(0)=0$, $$
(s^2+4)X(s) = \frac{s}{s^2+4}\left(1-e^{-2\pi s}\right) \quad \Longrightarrow \quad X(s) = \frac{s\left(1-e^{-2\pi s}\right)}{(s^2+4)^2}.
$$

**Step 3 — Invert.**

We use the standard resonance pair
$\mathcal{L}^{-1}\left[\dfrac{s}{(s^2+k^2)^2}\right] = \dfrac{t}{2k}\sin kt$
(which follows from $\mathcal{L}[t\sin kt] = \dfrac{2ks}{(s^2+k^2)^2}$).
With $k=2$: $$
\mathcal{L}^{-1}\left[\frac{s}{(s^2+4)^2}\right] = \frac{t}{4}\sin2t.
$$

By the shift theorem, the $e^{-2\pi s}$ term inverts to
$\dfrac{t-2\pi}{4}\sin\big(2(t-2\pi)\big)H(t-2\pi) = \dfrac{t-2\pi}{4}\sin(2t)H(t-2\pi)$
(again using the periodicity of $\sin2t$).

$$
x(t) = \frac{t}{4}\sin2t - \frac{t-2\pi}{4}\sin(2t)\,H(t-2\pi).
$$

**Step 4 — Simplify piecewise.** For $0\le t\le2\pi$ the Heaviside term
is off: $$
x(t) = \frac{t}{4}\sin2t.
$$ For $t>2\pi$, both terms are active and combine: $$
x(t) = \frac{\sin2t}{4}\Big[t-(t-2\pi)\Big] = \frac{\sin2t}{4}\cdot2\pi = \frac{\pi}{2}\sin2t.
$$

$$
\boxed{x(t) = \begin{cases} \dfrac{t}{4}\sin2t, & 0\le t\le2\pi \\[4pt] \dfrac{\pi}{2}\sin2t, & t>2\pi. \end{cases}}
$$

> **Tip**
>
> This is a classic **resonance-then-release** scenario: because the
> forcing frequency matches the natural frequency ($\omega=2=\sqrt4$),
> the amplitude grows *linearly* in $t$ while the forcing is on — the
> hallmark of pure resonance. The moment the forcing shuts off at
> $t=2\pi$, the growing envelope “freezes” at whatever amplitude it had
> reached ($\pi/2$), and the system continues forever after as an
> undamped, constant-amplitude oscillation at the natural frequency.

In [51]:
def x_closed(tt):
    return np.where(tt <= 2*np.pi, (tt/4)*np.sin(2*tt), (np.pi/2)*np.sin(2*tt))

t_vals = np.linspace(0, 4*np.pi, 1000)
x_vals = x_closed(t_vals)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(t_vals, x_vals, color='steelblue', lw=2)
ax.axvline(2*np.pi, color='gray', linestyle='--', lw=1, label=r'$t=2\pi$ (forcing turns off)')
ax.axhline(np.pi/2, color='gray', linestyle=':', lw=1)
ax.axhline(-np.pi/2, color='gray', linestyle=':', lw=1)
ax.set_xlabel('$t$')
ax.set_ylabel('$x(t)$')
ax.set_title(r"Solution of $x''+4x=\cos2t\,[1-H(t-2\pi)]$, $x(0)=x'(0)=0$")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [52]:
# Independent numerical check: integrate the ODE directly and compare to the closed form
def f_forcing(tt):
    return np.cos(2*tt) if tt <= 2*np.pi else 0.0

def rhs(tt, y):
    x1, x2 = y
    return [x2, -4*x1 + f_forcing(tt)]

sol = solve_ivp(rhs, [0, 4*np.pi], [0, 0], max_step=0.001, dense_output=True)

for tv in [1, 3, 5, 2*np.pi, 2*np.pi + 0.5, 8, 10]:
    num = sol.sol(tv)[0]
    closed = x_closed(tv)
    print(f"t={tv:.4f}: numeric={num:.6f}, closed-form={closed:.6f}, diff={abs(num-closed):.2e}")

t=1.0000: numeric=0.227324, closed-form=0.227324, diff=3.33e-16
t=3.0000: numeric=-0.209562, closed-form=-0.209562, diff=2.50e-16
t=5.0000: numeric=-0.680026, closed-form=-0.680026, diff=2.00e-15
t=6.2832: numeric=-0.000000, closed-form=-0.000000, diff=9.85e-10
t=6.7832: numeric=1.321782, closed-form=1.321780, diff=2.46e-06
t=8.0000: numeric=-0.452238, closed-form=-0.452237, diff=8.44e-07
t=10.0000: numeric=1.434054, closed-form=1.434051, diff=2.67e-06

> **Note**
>
> Direct numerical integration of the ODE matches the closed-form
> piecewise solution at every test point (to within numerical
> integration tolerance), confirming the Laplace-transform derivation.

------------------------------------------------------------------------

#### Exercise 14

Solve $x' = -x + H(t-1) - H(t-2),\ x(0)=1$.

**By Hand.**

$$
sX(s)-1 = -X(s) + \frac{e^{-s}}{s} - \frac{e^{-2s}}{s} \quad \Longrightarrow \quad X(s)(s+1) = 1+\frac{e^{-s}}{s}-\frac{e^{-2s}}{s}.
$$

$$
X(s) = \frac{1}{s+1} + \frac{e^{-s}}{s(s+1)} - \frac{e^{-2s}}{s(s+1)}.
$$

As in Exercise 6(a) above,
$\mathcal{L}^{-1}\left[\dfrac{1}{s(s+1)}\right] = 1-e^{-t}$. Applying
the shift theorem to each exponential term:

$$
\boxed{x(t) = e^{-t} + \left(1-e^{-(t-1)}\right)H(t-1) - \left(1-e^{-(t-2)}\right)H(t-2).}
$$

> **Tip**
>
> The forcing $H(t-1)-H(t-2)$ is itself a unit-height rectangular pulse
> on $[1,2)$ — structurally the same kind of input as Exercise 7’s pulse
> — so the response is the sum of the natural decay $e^{-t}$ plus the
> system’s reaction to “switching on” at $t=1$ and “switching back off”
> at $t=2$.

In [53]:
eq14 = sym.Eq(sym.diff(x(t), t), -x(t) + H(t-1) - H(t-2))
sol14 = sym.dsolve(eq14, x(t), ics={x(0): 1})
print(sym.simplify(sol14))

mine14 = sym.exp(-t) + (1 - sym.exp(-(t-1)))*H(t-1) - (1 - sym.exp(-(t-2)))*H(t-2)
print("Boxed formula:", mine14)

Eq(x(t), (-exp(t)*Heaviside(t - 2) + exp(t)*Heaviside(t - 1) + exp(2)*Heaviside(t - 2) - E*Heaviside(t - 1) + 1)*exp(-t))
Boxed formula: (1 - exp(1 - t))*Heaviside(t - 1) - (1 - exp(2 - t))*Heaviside(t - 2) + exp(-t)

> **Note**
>
> SymPy’s solution, when its terms are regrouped by Heaviside factor, is
> algebraically identical to the boxed formula above.

In [54]:
def x14(tt):
    base = np.exp(-tt)
    pulse_on = np.where(tt >= 1, 1 - np.exp(-(tt - 1)), 0.0)
    pulse_off = np.where(tt >= 2, 1 - np.exp(-(tt - 2)), 0.0)
    return base + pulse_on - pulse_off

t_vals = np.linspace(0, 6, 1000)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(t_vals, x14(t_vals), color='steelblue', lw=2)
ax.axvline(1, color='gray', linestyle='--', lw=1)
ax.axvline(2, color='gray', linestyle='--', lw=1)
ax.set_xlabel('$t$')
ax.set_ylabel('$x(t)$')
ax.set_title(r"Solution of $x'=-x+H(t-1)-H(t-2)$, $x(0)=1$")
plt.tight_layout()
plt.show()

------------------------------------------------------------------------

## Homework 7

### Section 3.3 — The Convolution Property

#### Exercise 1: Compute the Following Convolutions

**(a) $\sin t * \cos t$.**

By the convolution theorem, $\mathcal{L}[f*g] = F(s)G(s)$. With
$F(s)=\mathcal{L}[\sin t]=\dfrac{1}{s^2+1}$ and
$G(s)=\mathcal{L}[\cos t]=\dfrac{s}{s^2+1}$: $$
\mathcal{L}[\sin t * \cos t] = \frac{s}{(s^2+1)^2}.
$$

Using the resonance pair
$\mathcal{L}^{-1}\left[\dfrac{s}{(s^2+k^2)^2}\right] = \dfrac{t}{2k}\sin kt$
with $k=1$:

$$
\boxed{\sin t * \cos t = \frac{t}{2}\sin t.}
$$

**(c) $t * t^2$.**

$\mathcal{L}[t]=\dfrac{1}{s^2}$, $\mathcal{L}[t^2]=\dfrac{2}{s^3}$, so
$\mathcal{L}[t*t^2] = \dfrac{2}{s^5}$. Using
$\mathcal{L}^{-1}\left[\dfrac{n!}{s^{n+1}}\right]=t^n$ with $n=4$ (so
$\mathcal{L}^{-1}[1/s^5]=t^4/4!$):

$$
\boxed{t * t^2 = \frac{2t^4}{4!} = \frac{t^4}{12}.}
$$

In [55]:
t, tau, s = sym.symbols('t tau s', positive=True)

print("1a:", sym.simplify(sym.integrate(sym.sin(tau)*sym.cos(t-tau), (tau, 0, t))))
print("1c:", sym.simplify(sym.integrate(tau*(t-tau)**2, (tau, 0, t))))

1a: t*sin(t)/2
1c: t**4/12

> **Note**
>
> Both convolution integrals, computed directly from the definition
> $(f*g)(t)=\int_0^tf(\tau)g(t-\tau)\,d\tau$, confirm the boxed formulas
> obtained via the convolution theorem.

------------------------------------------------------------------------

#### Exercise 3

**(a)** Using Laplace transforms and the convolution property, find the
general solution of $x'-ax=f(t),\ x(0)=x_0$.

**By Hand.** Taking the Laplace transform: $$
sX(s)-x_0-aX(s) = F(s) \quad \Longrightarrow \quad X(s)(s-a) = x_0+F(s) \quad \Longrightarrow \quad X(s) = \frac{x_0}{s-a} + \frac{F(s)}{s-a}.
$$

Since $\mathcal{L}^{-1}\left[\dfrac{1}{s-a}\right]=e^{at}$, the first
term inverts to $x_0e^{at}$, and the second term is a product of
transforms, so it inverts to a convolution:

$$
\boxed{x(t) = x_0e^{at} + \int_0^te^{a(t-\tau)}f(\tau)\,d\tau.}
$$

**(b)** Solve using an integrating factor, and compare.

**By Hand.** Write $x'-ax=f(t)$ with integrating factor
$\mu(t)=e^{-at}$. Then $(e^{-at}x)' = e^{-at}f(t)$. Integrating from $0$
to $t$: $$
e^{-at}x(t) - x(0) = \int_0^te^{-a\tau}f(\tau)\,d\tau \quad \Longrightarrow \quad x(t) = x_0e^{at} + e^{at}\int_0^te^{-a\tau}f(\tau)\,d\tau.
$$

Pulling $e^{at}$ inside the integral (it doesn’t depend on $\tau$): $$
x(t) = x_0e^{at} + \int_0^te^{a(t-\tau)}f(\tau)\,d\tau.
$$

> **Tip**
>
> This is **identical** to the Laplace-transform result in part (a) —
> the convolution integral that falls out of the Laplace approach is
> exactly the particular-solution integral that the integrating-factor
> method produces directly. This is a nice concrete illustration of why
> the convolution theorem “works”: it’s really just packaging the
> variation-of-parameters/integrating-factor formula into transform
> language.

------------------------------------------------------------------------

#### Exercise 6

Solve the initial value problem $x''-\omega^2x=f(t),\ x(0)=x'(0)=0$.

**By Hand.**

$$
s^2X(s)-\omega^2X(s) = F(s) \quad \Longrightarrow \quad X(s) = \frac{F(s)}{s^2-\omega^2}.
$$

Partial fractions:
$\dfrac{1}{s^2-\omega^2} = \dfrac{1}{2\omega}\left(\dfrac{1}{s-\omega}-\dfrac{1}{s+\omega}\right)$,
whose inverse is
$\dfrac{1}{2\omega}\left(e^{\omega t}-e^{-\omega t}\right) = \dfrac{\sinh\omega t}{\omega}$.

$$
\boxed{x(t) = \frac{1}{\omega}\int_0^tf(\tau)\sinh\big(\omega(t-\tau)\big)\,d\tau.}
$$

> **Tip**
>
> Compare this to Example 3.25 in the text, which solves $x''+k^2x=f(t)$
> (oscillatory case) and gets the structurally identical formula with
> $\sin k(t-\tau)/k$ in place of $\sinh\omega(t-\tau)/\omega$ —
> replacing $k^2$ with $-\omega^2$ turns sines into hyperbolic sines,
> exactly as the characteristic roots change from imaginary ($\pm ik$)
> to real ($\pm\omega$).

------------------------------------------------------------------------

#### Exercise 7

Solve the initial value problem $x''-4x=1-H(t-1),\ x(0)=x'(0)=0$.

**By Hand.**

$$
(s^2-4)X(s) = \mathcal{L}[1-H(t-1)] = \frac1s - \frac{e^{-s}}{s}.
$$

$$
X(s) = \frac{1}{s(s^2-4)} - \frac{e^{-s}}{s(s^2-4)}.
$$

Partial fractions:
$\dfrac{1}{s(s-2)(s+2)} = -\dfrac{1}{4s}+\dfrac{1}{8(s-2)}+\dfrac{1}{8(s+2)}$,
whose inverse is $$
g(t) := \mathcal{L}^{-1}\left[\frac{1}{s(s^2-4)}\right] = -\frac14+\frac18e^{2t}+\frac18e^{-2t} = \frac14\big(\cosh2t-1\big).
$$

By the shift theorem, $x(t) = g(t) - g(t-1)H(t-1)$:

$$
\boxed{x(t) = \frac14\big(\cosh2t-1\big) - \frac14\big(\cosh\big(2(t-1)\big)-1\big)H(t-1).}
$$

In [56]:
x = sym.Function('x')
H = sym.Heaviside

eq7 = sym.Eq(sym.diff(x(t),t,2) - 4*x(t), 1 - H(t-1))
sol7 = sym.dsolve(eq7, x(t), ics={x(0): 0, x(t).diff(t).subs(t, 0): 0})

mine7 = sym.Rational(1,4)*(sym.cosh(2*t)-1) - sym.Rational(1,4)*(sym.cosh(2*(t-1))-1)*H(t-1)

# Numerically compare SymPy's (messier) closed form to the boxed formula
for tv in [0.3, 0.9, 1.5, 3]:
    a = float(sol7.rhs.subs(t, tv))
    b = float(mine7.subs(t, tv))
    print(f"t={tv}: dsolve={a:.6f}, boxed formula={b:.6f}, diff={abs(a-b):.2e}")

t=0.3: dsolve=0.046366, boxed formula=0.046366, diff=1.39e-17
t=0.9: dsolve=0.526868, boxed formula=0.526868, diff=1.11e-16
t=1.5: dsolve=2.131145, boxed formula=2.131145, diff=0.00e+00
t=3: dsolve=43.601851, boxed formula=43.601851, diff=0.00e+00

> **Note**
>
> SymPy’s `dsolve` output is algebraically messier (it doesn’t
> automatically recognize $\cosh$) but numerically agrees with the boxed
> formula at every test point.

------------------------------------------------------------------------

#### Exercise 8

Express the solution of $x''+3x'+2x=e^{-4t},\ x(0)=x'(0)=0$ as a
convolution integral.

**By Hand.**

$$
(s^2+3s+2)X(s) = \frac{1}{s+4} \quad \Longrightarrow \quad X(s) = \frac{1}{(s+1)(s+2)(s+4)}.
$$

The **transfer function** of the system (homogeneous part) is
$\dfrac{1}{(s+1)(s+2)}$, whose inverse — the **impulse response** — is,
via partial fractions
$\dfrac{1}{(s+1)(s+2)} = \dfrac{1}{s+1}-\dfrac{1}{s+2}$: $$
g(t) = e^{-t}-e^{-2t}.
$$

Since the forcing $f(t)=e^{-4t}$ has $F(s)=\dfrac{1}{s+4}$, and
$X(s)=G(s)F(s)$, the solution is the convolution $x=g*f$:

$$
\boxed{x(t) = \int_0^t\left(e^{-\tau}-e^{-2\tau}\right)e^{-4(t-\tau)}\,d\tau.}
$$

In [57]:
tau = sym.symbols('tau', positive=True)
g = sym.exp(-tau) - sym.exp(-2*tau)
conv8 = sym.integrate(g*sym.exp(-4*(t-tau)), (tau, 0, t))
print("Evaluated convolution:", sym.simplify(conv8))

eq8 = sym.Eq(sym.diff(x(t),t,2) + 3*sym.diff(x(t),t) + 2*x(t), sym.exp(-4*t))
sol8 = sym.dsolve(eq8, x(t), ics={x(0): 0, x(t).diff(t).subs(t, 0): 0})
print("dsolve:", sym.simplify(sol8))

Evaluated convolution: ((2*exp(t) - 3)*exp(2*t) + 1)*exp(-4*t)/6
dsolve: Eq(x(t), (2*exp(3*t) - 3*exp(2*t) + 1)*exp(-4*t)/6)

> **Note**
>
> Evaluating the convolution integral explicitly gives
> $x(t) = \dfrac{2e^{3t}-3e^{2t}+1}{6}e^{-4t}$, which matches SymPy’s
> independent `dsolve` solution exactly — confirming the convolution
> integral is correct (even though the exercise only asks for the
> integral form, not its evaluation).

------------------------------------------------------------------------

#### Exercise 9

Express the solution of $x''+2x'+2x=\sin\omega t,\ x(0)=x'(0)=0$ as a
convolution integral.

**By Hand.**

$$
(s^2+2s+2)X(s) = \frac{\omega}{s^2+\omega^2}.
$$

Completing the square, $s^2+2s+2=(s+1)^2+1$. The transfer function
$\dfrac{1}{(s+1)^2+1}$ has impulse response (via the shift theorem, with
$\mathcal{L}^{-1}[1/(s^2+1)]=\sin t$): $$
g(t) = e^{-t}\sin t.
$$

Since $f(t)=\sin\omega t$, the solution is $x=g*f$:

$$
\boxed{x(t) = \int_0^te^{-\tau}\sin\tau\cdot\sin\big(\omega(t-\tau)\big)\,d\tau.}
$$

In [58]:
w = sym.symbols('omega', positive=True)
g9 = sym.exp(-tau)*sym.sin(tau)

eq9 = sym.Eq(sym.diff(x(t),t,2) + 2*sym.diff(x(t),t) + 2*x(t), sym.sin(w*t))
sol9 = sym.dsolve(eq9, x(t), ics={x(0): 0, x(t).diff(t).subs(t, 0): 0})

# Numerically compare the evaluated convolution to dsolve's closed form.
# Substitute each numeric omega into the integrand BEFORE integrating --
# integrating with omega left symbolic is extremely slow in SymPy.
for wv, tv in [(1, 1.0), (2, 0.5), (0.5, 2.0)]:
    conv_val = sym.integrate(g9*sym.sin(wv*(t-tau)), (tau, 0, t)).subs(t, tv)
    a = float(conv_val)
    b = float(sol9.rhs.subs({w: wv, t: tv}))
    print(f"omega={wv}, t={tv}: convolution={a:.6f}, dsolve={b:.6f}, diff={abs(a-b):.2e}")

omega=1, t=1.0: convolution=0.093592, dsolve=0.093592, diff=0.00e+00
omega=2, t=0.5: convolution=0.030563, dsolve=0.030563, diff=1.39e-17
omega=0.5, t=2.0: convolution=0.219406, dsolve=0.219406, diff=0.00e+00

> **Note**
>
> The convolution integral, evaluated numerically at several
> $(\omega,t)$ pairs, matches SymPy’s independently-derived closed-form
> solution in every case.

------------------------------------------------------------------------

#### Exercise 12

Use convolution to calculate
$\mathcal{L}^{-1}\left[\dfrac{1}{s^2(s^2+1)}\right]$.

**By Hand.**

Write
$\dfrac{1}{s^2(s^2+1)} = \dfrac{1}{s^2}\cdot\dfrac{1}{s^2+1} = F(s)G(s)$
with $F(s)=\mathcal{L}[t]$ and $G(s)=\mathcal{L}[\sin t]$. By the
convolution theorem, $$
\mathcal{L}^{-1}\left[\frac{1}{s^2(s^2+1)}\right] = t*\sin t = \int_0^t\tau\sin(t-\tau)\,d\tau.
$$

Expanding $\sin(t-\tau)=\sin t\cos\tau-\cos t\sin\tau$ and using
$\int_0^t\tau\cos\tau\,d\tau = t\sin t+\cos t-1$ and
$\int_0^t\tau\sin\tau\,d\tau=-t\cos t+\sin t$: $$
t*\sin t = \sin t\,(t\sin t+\cos t-1) - \cos t\,(-t\cos t+\sin t) = t(\sin^2t+\cos^2t) - \sin t = t-\sin t.
$$

$$
\boxed{\mathcal{L}^{-1}\left[\frac{1}{s^2(s^2+1)}\right] = t-\sin t.}
$$

In [59]:
conv12 = sym.integrate(tau*sym.sin(t-tau), (tau, 0, t))
print("Convolution t * sin t =", sym.simplify(conv12))

F12 = 1/(s**2*(s**2+1))
f12 = sym.inverse_laplace_transform(F12, s, t)
print("Direct inverse transform:", sym.simplify(f12))

Convolution t * sin t = t - sin(t)
Direct inverse transform: t - sin(t)

> **Note**
>
> Both the hand-computed convolution and SymPy’s direct
> inverse-transform computation give $t-\sin t$.

------------------------------------------------------------------------

#### Exercise 13

Use convolution to compute
$\mathcal{L}^{-1}\left[\dfrac{1}{(s^2+1)^2}\right]$.

**By Hand.**

Write
$\dfrac{1}{(s^2+1)^2} = \dfrac{1}{s^2+1}\cdot\dfrac{1}{s^2+1} = F(s)G(s)$
with $F(s)=G(s)=\mathcal{L}[\sin t]$. By the convolution theorem, $$
\mathcal{L}^{-1}\left[\frac{1}{(s^2+1)^2}\right] = \sin t*\sin t = \int_0^t\sin\tau\sin(t-\tau)\,d\tau.
$$

Using the product-to-sum identity
$\sin\tau\sin(t-\tau) = \dfrac12\big[\cos(2\tau-t)-\cos t\big]$: $$
\int_0^t\sin\tau\sin(t-\tau)\,d\tau = \frac12\int_0^t\cos(2\tau-t)\,d\tau - \frac{\cos t}{2}\int_0^td\tau.
$$

The first integral evaluates to
$\dfrac12\left[\dfrac{\sin(2\tau-t)}{2}\right]_0^t = \dfrac14\big[\sin t-\sin(-t)\big] = \dfrac{\sin t}{2}$,
and the second is simply $\dfrac{t\cos t}{2}$. Subtracting:

$$
\boxed{\mathcal{L}^{-1}\left[\frac{1}{(s^2+1)^2}\right] = \frac{\sin t - t\cos t}{2}.}
$$

In [60]:
conv13 = sym.integrate(sym.sin(tau)*sym.sin(t-tau), (tau, 0, t))
print("Convolution sin t * sin t =", sym.simplify(conv13))

F13 = 1/(s**2+1)**2
f13 = sym.inverse_laplace_transform(F13, s, t)
print("Direct inverse transform:", sym.simplify(f13))

Convolution sin t * sin t = -t*cos(t)/2 + sin(t)/2
Direct inverse transform: -t*cos(t)/2 + sin(t)/2

> **Note**
>
> Both the hand-computed convolution and SymPy’s direct
> inverse-transform computation give $\dfrac{\sin t - t\cos t}{2}$.

> **Tip**
>
> Compare with Exercise 1(a) above:
> $\sin t*\cos t = \dfrac{t}{2}\sin t$, while
> $\sin t*\sin t = \dfrac{\sin t-t\cos t}{2}$. Both involve the same
> “resonance” mechanism — convolving a sinusoid with itself (or a
> phase-shifted copy of itself) at matching frequency produces a term
> that grows linearly in $t$, the time-domain signature of the repeated
> pole at $s=\pm i$ in $(s^2+1)^2$.

------------------------------------------------------------------------

### Section 3.4 — Impulsive Sources

#### Exercise 1

Compute $\displaystyle\int_0^\infty e^{-2(t-3)^2}\delta_4(t)\,dt$.

**By Hand.** Here $\delta_4(t)$ denotes $\delta(t-4)$. By the sifting
property $\int_{-\infty}^\infty f(t)\delta(t-a)\,dt=f(a)$, with
$f(t)=e^{-2(t-3)^2}$ and $a=4$:

$$
\boxed{\int_0^\infty e^{-2(t-3)^2}\delta_4(t)\,dt = e^{-2(4-3)^2} = e^{-2} \approx 0.1353.}
$$

In [61]:
val = sym.integrate(sym.exp(-2*(t-3)**2)*sym.DiracDelta(t-4), (t, 0, sym.oo))
print("Integral =", val, "=", sym.N(val))
print("e^(-2) =", sym.N(sym.exp(-2)))

Integral = exp(-2) = 0.135335283236613
e^(-2) = 0.135335283236613

> **Note**
>
> SymPy’s symbolic integration of the Dirac delta confirms the
> sifting-property result exactly.

------------------------------------------------------------------------

#### Exercise 2

Solve the initial value problem $x'+3x=\delta_1(t)+H(t-4),\ x(0)=1$.
Sketch the solution.

**By Hand.**

$$
sX(s)-1+3X(s) = e^{-s} + \frac{e^{-4s}}{s} \quad \Longrightarrow \quad X(s) = \frac{1}{s+3} + \frac{e^{-s}}{s+3} + \frac{e^{-4s}}{s(s+3)}.
$$

Using $\mathcal{L}^{-1}\left[\dfrac1{s+3}\right]=e^{-3t}$ and, by
partial fractions,
$\mathcal{L}^{-1}\left[\dfrac{1}{s(s+3)}\right] = \dfrac13\left(1-e^{-3t}\right)$,
together with the shift theorem:

$$
\boxed{x(t) = e^{-3t} + e^{-3(t-1)}H(t-1) + \frac13\left(1-e^{-3(t-4)}\right)H(t-4).}
$$

In [62]:
eq2 = sym.Eq(sym.diff(x(t), t) + 3*x(t), sym.DiracDelta(t-1) + H(t-4))
sol2 = sym.dsolve(eq2, x(t), ics={x(0): 1})
print(sym.simplify(sol2))

Eq(x(t), (exp(3*t)*Heaviside(t - 4) - exp(12)*Heaviside(t - 4) + 3*exp(3)*Heaviside(t - 1) + 3)*exp(-3*t)/3)

> **Note**
>
> SymPy’s result, regrouped by Heaviside factor, is algebraically
> identical to the boxed formula.

In [63]:
def x2_func(tt):
    tt = np.asarray(tt, dtype=float)
    out = np.exp(-3*tt)
    out = out + np.where(tt >= 1, np.exp(-3*(tt-1)), 0.0)
    out = out + np.where(tt >= 4, (1/3)*(1-np.exp(-3*(tt-4))), 0.0)
    return out

t_vals = np.linspace(0, 6, 2000)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(t_vals, x2_func(t_vals), color='steelblue', lw=2)
ax.axvline(1, color='gray', linestyle='--', lw=1, label='impulse at $t=1$')
ax.axvline(4, color='gray', linestyle=':', lw=1, label='step on at $t=4$')
ax.set_xlabel('$t$')
ax.set_ylabel('$x(t)$')
ax.set_title(r"$x'+3x=\delta_1(t)+H(t-4)$, $x(0)=1$")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

------------------------------------------------------------------------

#### Exercise 3

Solve the initial value problem $x''-x=\delta_5(t),\ x(0)=x'(0)=0$.
Sketch the solution.

**By Hand.**

$$
(s^2-1)X(s) = e^{-5s} \quad \Longrightarrow \quad X(s) = \frac{e^{-5s}}{s^2-1}.
$$

Using $\mathcal{L}^{-1}\left[\dfrac{1}{s^2-1}\right] = \sinh t$ and the
shift theorem:

$$
\boxed{x(t) = \sinh(t-5)\,H(t-5).}
$$

In [64]:
eq3 = sym.Eq(sym.diff(x(t), t, 2) - x(t), sym.DiracDelta(t-5))
sol3 = sym.dsolve(eq3, x(t), ics={x(0): 0, x(t).diff(t).subs(t, 0): 0})
print(sym.simplify(sol3))

Eq(x(t), (exp(2*t) - exp(10))*exp(-t - 5)*Heaviside(t - 5)/2)

> **Note**
>
> SymPy’s result $\dfrac{e^{2t}-e^{10}}{2e^{t+5}}H(t-5)$ simplifies
> exactly to $\sinh(t-5)H(t-5)$.

In [65]:
def x3_func(tt):
    tt = np.asarray(tt, dtype=float)
    return np.where(tt >= 5, np.sinh(tt-5), 0.0)

t_vals = np.linspace(0, 7, 2000)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(t_vals, x3_func(t_vals), color='steelblue', lw=2)
ax.axvline(5, color='gray', linestyle='--', lw=1, label='impulse at $t=5$')
ax.set_xlabel('$t$')
ax.set_ylabel('$x(t)$')
ax.set_title(r"$x''-x=\delta_5(t)$, $x(0)=x'(0)=0$")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

------------------------------------------------------------------------

#### Exercise 4

Solve the initial value problem $x''+x=\delta_2(t),\ x(0)=x'(0)=0$.
Sketch the solution.

**By Hand.**

$$
(s^2+1)X(s) = e^{-2s} \quad \Longrightarrow \quad X(s) = \frac{e^{-2s}}{s^2+1}.
$$

Using $\mathcal{L}^{-1}\left[\dfrac1{s^2+1}\right]=\sin t$ and the shift
theorem:

$$
\boxed{x(t) = \sin(t-2)\,H(t-2).}
$$

In [66]:
eq4 = sym.Eq(sym.diff(x(t), t, 2) + x(t), sym.DiracDelta(t-2))
sol4 = sym.dsolve(eq4, x(t), ics={x(0): 0, x(t).diff(t).subs(t, 0): 0})
print(sym.simplify(sol4))

Eq(x(t), sin(t - 2)*Heaviside(t - 2))

> **Note**
>
> SymPy confirms $x(t)=\sin(t-2)H(t-2)$ exactly.

In [67]:
def x4_func(tt):
    tt = np.asarray(tt, dtype=float)
    return np.where(tt >= 2, np.sin(tt-2), 0.0)

t_vals = np.linspace(0, 12, 2000)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(t_vals, x4_func(t_vals), color='steelblue', lw=2)
ax.axvline(2, color='gray', linestyle='--', lw=1, label='impulse at $t=2$')
ax.set_xlabel('$t$')
ax.set_ylabel('$x(t)$')
ax.set_title(r"$x''+x=\delta_2(t)$, $x(0)=x'(0)=0$")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

> **Tip**
>
> Exercises 3 and 4 are a nice contrast: both impulses inject the *same*
> kind of instantaneous “kick” (a unit jump in velocity at the moment of
> impact), but the underlying dynamics are completely different —
> $x''-x=0$ is unstable (exponential growth after the kick), while
> $x''+x=0$ is neutrally stable (perpetual oscillation after the kick).

------------------------------------------------------------------------

#### Exercise 5

Invert the transform $F(s) = \dfrac{e^{-2s}}{s} + e^{-3s}$.

**By Hand.**

Each term is handled separately:

- $\mathcal{L}^{-1}\left[\dfrac{e^{-2s}}{s}\right] = H(t-2)$, by the
  shift theorem applied to $\mathcal{L}^{-1}[1/s]=1$.
- $\mathcal{L}^{-1}\left[e^{-3s}\right] = \delta(t-3)$, since
  $\mathcal{L}[\delta(t-a)] = e^{-as}$, so a bare exponential $e^{-as}$
  (with no other factor) inverts to a pure delta spike.

$$
\boxed{f(t) = H(t-2) + \delta(t-3).}
$$

In [68]:
F5 = sym.exp(-2*s)/s + sym.exp(-3*s)
f5 = sym.inverse_laplace_transform(F5, s, t)
print(f5)

DiracDelta(t - 3) + Heaviside(t - 2)

> **Note**
>
> SymPy confirms $f(t) = \delta(t-3) + H(t-2)$.

------------------------------------------------------------------------

### Problem Set 3 — $2\times2$ Matrices and Linear Systems

This set is practice with the linear algebra needed for systems of ODEs
(Notes 9, §4.2 of Logan), in preparation for Exam 2.

#### (a) Converting Second-Order Equations to First-Order Systems

**(i) $x''+4x'+3x=0$ (linear).**

Let $x_1=x,\ x_2=x'$. Then $x_1'=x_2$, and
$x_2'=x''=-4x'-3x=-4x_2-3x_1$.

$$
\boxed{x_1'=x_2, \qquad x_2'=-3x_1-4x_2.}
$$

In matrix/vector form $\mathbf{x}'=A\mathbf{x}$ with
$\mathbf{x}=\begin{pmatrix}x_1\\x_2\end{pmatrix}$:

$$
\boxed{\mathbf{x}' = \begin{pmatrix}0&1\\-3&-4\end{pmatrix}\mathbf{x}.}
$$

**(ii) $x''+x'+x^3=0$ (nonlinear).**

Let $x_1=x,\ x_2=x'$. Then $x_1'=x_2$, and
$x_2'=x''=-x'-x^3=-x_2-x_1^3$.

$$
\boxed{x_1'=x_2, \qquad x_2'=-x_1^3-x_2.}
$$

**Why this can’t be written as $\mathbf{x}'=A\mathbf{x}$.** A matrix
equation $\mathbf{x}'=A\mathbf{x}$ requires every component on the
right-hand side to be a **linear combination** of $x_1$ and $x_2$ —
i.e. each row of $A\mathbf{x}$ has the form $a_{i1}x_1+a_{i2}x_2$. Here
the second equation contains the term $-x_1^3$, which is **cubic**, not
linear, in $x_1$. No constant matrix $A$ can reproduce a cubic term by
multiplying $\mathbf{x}$, so the system is inherently nonlinear and
cannot be put in the form $\mathbf{x}'=A\mathbf{x}$.

------------------------------------------------------------------------

#### (b) System in Matrix/Vector Form

Write $x_1'=3x_1-x_2,\ x_2'=2x_1+4x_2$ in the form
$\mathbf{x}'=A\mathbf{x}$.

Reading the coefficients directly off each equation:

$$
\boxed{\mathbf{x}' = \begin{pmatrix}3&-1\\2&4\end{pmatrix}\mathbf{x}.}
$$

------------------------------------------------------------------------

#### (c) Matrix Products

Let
$J=\begin{pmatrix}2&3\\-1&0\end{pmatrix},\ K=\begin{pmatrix}1&-2\\4&1\end{pmatrix},\ v=\begin{pmatrix}2\\-1\end{pmatrix}$.

**$JK$:** $$
JK = \begin{pmatrix}2&3\\-1&0\end{pmatrix}\begin{pmatrix}1&-2\\4&1\end{pmatrix} = \begin{pmatrix}2(1)+3(4) & 2(-2)+3(1)\\-1(1)+0(4) & -1(-2)+0(1)\end{pmatrix} = \boxed{\begin{pmatrix}14&-1\\-1&2\end{pmatrix}.}
$$

**$KJ$:** $$
KJ = \begin{pmatrix}1&-2\\4&1\end{pmatrix}\begin{pmatrix}2&3\\-1&0\end{pmatrix} = \begin{pmatrix}1(2)+(-2)(-1) & 1(3)+(-2)(0)\\4(2)+1(-1) & 4(3)+1(0)\end{pmatrix} = \boxed{\begin{pmatrix}4&3\\7&12\end{pmatrix}.}
$$

Note $JK\ne KJ$, confirming matrix multiplication is **not commutative**
in general.

**$Jv$:** $$
Jv = \begin{pmatrix}2&3\\-1&0\end{pmatrix}\begin{pmatrix}2\\-1\end{pmatrix} = \begin{pmatrix}2(2)+3(-1)\\-1(2)+0(-1)\end{pmatrix} = \boxed{\begin{pmatrix}1\\-2\end{pmatrix}.}
$$

In [69]:
J = sym.Matrix([[2, 3], [-1, 0]])
K = sym.Matrix([[1, -2], [4, 1]])
v = sym.Matrix([2, -1])

print("JK =", J*K)
print("KJ =", K*J)
print("Jv =", J*v)

JK = Matrix([[14, -1], [-1, 2]])
KJ = Matrix([[4, 3], [7, 12]])
Jv = Matrix([[1], [-2]])

> **Note**
>
> All three products match the boxed results.

------------------------------------------------------------------------

#### (d) Determinants

**(i) $P=\begin{pmatrix}3&5\\1&2\end{pmatrix}$:**
$\det P = 3(2)-5(1) = 6-5 = \boxed{1}.$

**(ii) $Q=\begin{pmatrix}-2&4\\3&-1\end{pmatrix}$:**
$\det Q = (-2)(-1)-4(3) = 2-12 = \boxed{-10}.$

**(iii) $R=\begin{pmatrix}4&6\\2&3\end{pmatrix}$:**
$\det R = 4(3)-6(2) = 12-12 = \boxed{0}.$

In [70]:
P = sym.Matrix([[3, 5], [1, 2]])
Q = sym.Matrix([[-2, 4], [3, -1]])
R = sym.Matrix([[4, 6], [2, 3]])

print("det P =", P.det())
print("det Q =", Q.det())
print("det R =", R.det())

det P = 1
det Q = -10
det R = 0

> **Tip**
>
> $\det R = 0$ means $R$ is **singular**: its rows (and columns) are
> linearly dependent. Indeed, row 1 is exactly twice row 2:
> $(4,6) = 2\cdot(2,3)$.

------------------------------------------------------------------------

#### (e) Linear Independence of Vector Pairs

Two vectors in $\mathbb{R}^2$ are linearly independent exactly when the
matrix formed by using them as columns (or rows) has **nonzero
determinant**.

**(i)
$\begin{pmatrix}3\\1\end{pmatrix},\begin{pmatrix}1\\2\end{pmatrix}$:**
$\det\begin{pmatrix}3&1\\1&2\end{pmatrix} = 6-1 = 5 \ne 0$. $$
\boxed{\text{Linearly independent.}}
$$

**(ii)
$\begin{pmatrix}-2\\4\end{pmatrix},\begin{pmatrix}1\\-2\end{pmatrix}$:**
$\det\begin{pmatrix}-2&1\\4&-2\end{pmatrix} = 4-4 = 0$. $$
\boxed{\text{Linearly dependent}} \quad \left(\text{note } \begin{pmatrix}-2\\4\end{pmatrix} = -2\begin{pmatrix}1\\-2\end{pmatrix}\right).
$$

**(iii)
$\begin{pmatrix}5\\2\end{pmatrix},\begin{pmatrix}-1\\3\end{pmatrix}$:**
$\det\begin{pmatrix}5&-1\\2&3\end{pmatrix} = 15-(-2) = 17 \ne 0$. $$
\boxed{\text{Linearly independent.}}
$$

In [71]:
pairs = [
    (sym.Matrix([3, 1]), sym.Matrix([1, 2])),
    (sym.Matrix([-2, 4]), sym.Matrix([1, -2])),
    (sym.Matrix([5, 2]), sym.Matrix([-1, 3])),
]
for u, w in pairs:
    M = sym.Matrix.hstack(u, w)
    status = "independent" if M.det() != 0 else "dependent"
    print(f"{u.T.tolist()[0]}, {w.T.tolist()[0]} -> det = {M.det()}  ({status})")

[3, 1], [1, 2] -> det = 5  (independent)
[-2, 4], [1, -2] -> det = 0  (dependent)
[5, 2], [-1, 3] -> det = 17  (independent)

> **Note**
>
> All three determinant computations, and the resulting
> independence/dependence conclusions, match the boxed results.

------------------------------------------------------------------------

#### (f) Eigenvalues and Eigenvectors

**(i) $N_1=\begin{pmatrix}5&-2\\-2&2\end{pmatrix}$.**

The characteristic equation is $\det(N_1-\lambda I)=0$: $$
(5-\lambda)(2-\lambda)-(-2)(-2) = \lambda^2-7\lambda+6 = (\lambda-1)(\lambda-6) = 0 \quad \Longrightarrow \quad \lambda=1,\ 6.
$$

**For $\lambda=1$:** $N_1-I=\begin{pmatrix}4&-2\\-2&1\end{pmatrix}$. The
equation $4v_1-2v_2=0$ gives $v_2=2v_1$, so an eigenvector is
$\begin{pmatrix}1\\2\end{pmatrix}$.

**For $\lambda=6$:** $N_1-6I=\begin{pmatrix}-1&-2\\-2&-4\end{pmatrix}$.
The equation $-v_1-2v_2=0$ gives $v_1=-2v_2$, so an eigenvector is
$\begin{pmatrix}-2\\1\end{pmatrix}$.

$$
\boxed{\lambda_1=1,\ \mathbf{v}_1=\begin{pmatrix}1\\2\end{pmatrix}; \qquad \lambda_2=6,\ \mathbf{v}_2=\begin{pmatrix}-2\\1\end{pmatrix}.}
$$

**(ii) $N_2=\begin{pmatrix}2&4\\1&-1\end{pmatrix}$.**

$$
(2-\lambda)(-1-\lambda)-4(1) = \lambda^2-\lambda-6 = (\lambda-3)(\lambda+2) = 0 \quad \Longrightarrow \quad \lambda=3,\ -2.
$$

**For $\lambda=3$:** $N_2-3I=\begin{pmatrix}-1&4\\1&-4\end{pmatrix}$.
The equation $-v_1+4v_2=0$ gives $v_1=4v_2$, so an eigenvector is
$\begin{pmatrix}4\\1\end{pmatrix}$.

**For $\lambda=-2$:** $N_2+2I=\begin{pmatrix}4&4\\1&1\end{pmatrix}$. The
equation $4v_1+4v_2=0$ gives $v_1=-v_2$, so an eigenvector is
$\begin{pmatrix}-1\\1\end{pmatrix}$.

$$
\boxed{\lambda_1=3,\ \mathbf{v}_1=\begin{pmatrix}4\\1\end{pmatrix}; \qquad \lambda_2=-2,\ \mathbf{v}_2=\begin{pmatrix}-1\\1\end{pmatrix}.}
$$

In [72]:
N1 = sym.Matrix([[5, -2], [-2, 2]])
N2 = sym.Matrix([[2, 4], [1, -1]])

print("N1 eigenvects (eigenvalue, algebraic multiplicity, [eigenvectors]):")
print(N1.eigenvects())
print()
print("N2 eigenvects (eigenvalue, algebraic multiplicity, [eigenvectors]):")
print(N2.eigenvects())

N1 eigenvects (eigenvalue, algebraic multiplicity, [eigenvectors]):
[(1, 1, [Matrix([
[1/2],
[  1]])]), (6, 1, [Matrix([
[-2],
[ 1]])])]

N2 eigenvects (eigenvalue, algebraic multiplicity, [eigenvectors]):
[(-2, 1, [Matrix([
[-1],
[ 1]])]), (3, 1, [Matrix([
[4],
[1]])])]

> **Note**
>
> SymPy confirms both eigenvalue/eigenvector pairs for $N_1$ and $N_2$
> (its eigenvector for $N_1$’s $\lambda=1$ is
> $\begin{pmatrix}1/2\\1\end{pmatrix}$, a scalar multiple of the boxed
> $\begin{pmatrix}1\\2\end{pmatrix}$ — any nonzero scalar multiple of an
> eigenvector is itself a valid eigenvector).

> **Tip**
>
> Both $N_1$ and $N_2$ are diagonalizable (two linearly independent
> eigenvectors each), so for a system $\mathbf{x}'=N\mathbf{x}$ built
> from either matrix, the general solution is a superposition of the
> pure exponential modes $e^{\lambda_1t}\mathbf{v}_1$ and
> $e^{\lambda_2t}\mathbf{v}_2$ — exactly the picture described in the
> assignment’s notes on eigenvalues as the “natural directions” of a
> linear system.

------------------------------------------------------------------------

------------------------------------------------------------------------

## References

> **Expand for Session Info**
>
> ``` python
> import sys # sys for system-specific parameters and functions
> print("Python version:", sys.version)
> print('\n'.join(f'{m.__name__}=={m.__version__}' for m in globals().values() if getattr(m, '__version__', None)))
> ```
>
>     Python version: 3.14.4 | packaged by conda-forge | (main, Apr  8 2026, 02:33:53) [Clang 20.1.8 ]
>     numpy==2.4.3
>     scipy==1.17.1
>     sympy==1.14.0
>     matplotlib==3.10.8

## Reuse

[![](http://mirrors.creativecommons.org/presskit/buttons/88x31/png/by-nc-sa.png?raw=1)](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode)

[CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)